# Liu2024 — Supervised Classifier Diagnostics

A sanity-check notebook. Before investing more in S-JEPA, this compares a small set of
**standard, interpretable supervised baselines** on the *same* Liu2024 data, preprocessing,
within-subject cross-validation, metrics, and artifact style as
`liu2024_source_mat_sjepa_prelocal_augmented_clean.ipynb` — but with **no dependence on
S-JEPA pretraining**.

**Baselines (kept deliberately simple):**
- `logreg_bandpower` — log band-power per channel + standardization + logistic regression
  (pure scikit-learn; the most transparent baseline).
- `csp_lda` — Common Spatial Patterns + LDA (the classic motor-imagery baseline; needs MNE).
- `fbcsp_lda` — filter-bank CSP + LDA (optional; a compact FBCSP).
- `eegnet` — a compact EEGNet CNN trained supervised (needs braindecode; modest epochs).
- `riemann_mdm` — covariance + minimum-distance-to-Riemannian-mean (optional; needs pyriemann).

**Conservative by construction:** augmentation is configurable but **disabled by default**;
preprocessing is carried verbatim from the reference (nothing changed silently); the
reference's prediction-balance loss penalty is set to **0** here so a collapsing model is not
hidden by the loss. Fold-safe throughout: feature scalers, CSP, and covariance estimators are
fit on the training split only.

## Diagnostic questions this notebook targets
1. Is the train/test split balanced? → per-fold class counts + an explicit balance check.
2. Are labels handled correctly? → label inventory, the {1,2}→{0,1} mapping, per-subject counts.
3. Subject- or fold-level collapse? → `collapse_rate`, `collapse_ratio`, majority class per fold.
4. Are some subjects consistently impossible? → per-subject balanced accuracy across all methods.
5. Does a simple supervised classifier beat the S-JEPA head? → optional load of a prior S-JEPA
   run's `global_metrics.json` into the comparison.
6. Does accuracy depend on the window start/time range? → a window-start sensitivity sweep.
7. Is the model predicting mostly one class? → prediction histograms + right-class rate.
8. Are probabilities calibrated or saturated? → confidence/entropy + a reliability curve.
9. Are failures tied to specific folds/subjects? → per-fold, per-subject tables and heatmaps.

# 1. Setup

In [1]:
import copy, os, re, sys, json, math, hashlib, random, builtins, platform, inspect
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.io import loadmat
from scipy import signal

from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print(f"[setup] matplotlib unavailable -> plots skipped: {exc}")

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, Subset
    HAVE_TORCH = True
except Exception as exc:
    HAVE_TORCH = False
    Dataset = object
    print(f"[setup] torch unavailable -> EEGNet path skipped: {exc}")

try:
    import mne
    mne.set_log_level("WARNING")
    HAVE_MNE = True
except Exception as exc:
    HAVE_MNE = False
    print(f"[setup] mne unavailable -> data loading + CSP skipped: {exc}")

try:
    from mne.decoding import CSP
    HAVE_CSP = True
except Exception as exc:
    HAVE_CSP = False
    print(f"[setup] mne.decoding.CSP unavailable -> CSP baselines skipped: {exc}")

try:
    from skorch.callbacks import EarlyStopping, EpochScoring
    from skorch.dataset import ValidSplit
    from braindecode import EEGClassifier
    from braindecode.models import EEGNetv4
    HAVE_BRAINDECODE = True
except Exception as exc:
    HAVE_BRAINDECODE = False
    print(f"[setup] braindecode/skorch unavailable -> EEGNet path skipped: {exc}")

try:
    from pyriemann.estimation import Covariances
    from pyriemann.classification import MDM
    HAVE_PYRIEMANN = True
except Exception as exc:
    HAVE_PYRIEMANN = False
    print(f"[setup] pyriemann unavailable -> Riemannian MDM baseline skipped: {exc}")

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("deps:", dict(torch=HAVE_TORCH, mne=HAVE_MNE, csp=HAVE_CSP,
                    braindecode=HAVE_BRAINDECODE, pyriemann=HAVE_PYRIEMANN, mpl=HAVE_MPL))


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


deps: {'torch': True, 'mne': True, 'csp': True, 'braindecode': True, 'pyriemann': True, 'mpl': True}


# 2. Configuration

## 2.1 Channel defaults (carried verbatim)

In [2]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


## 2.2 CONFIG

Reference CONFIG carried verbatim (identical preprocessing, splits, seeds), then extended
with: the baseline method registry, conservative supervised-training defaults for the EEGNet
path, the window-start sensitivity grid, and an optional path to a prior S-JEPA run for the
head-to-head comparison. Note the two **deliberate** conservative overrides flagged below.

In [3]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "baseline_sjepa_prelocal",
    "config_note": "Clean MNE-style preprocessing pipeline builder + S-JEPA hyperparameter controls.",

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ------------------------------------------------------------------
    # Source-domain preprocessing before creating MNE RawArray
    # ------------------------------------------------------------------
    "demean_mode": "none",      # none, trial_mean, baseline_window_mean
    "baseline_window_s": [0.0, 2.0],
    "detrend_mode": "none",                     # none, constant, linear
    "eog_correction": "none",                   # none, linear_regression

    # Robust source-domain clipping / winsorization. Use cautiously.
    "artifact_clip_mode": "none",               # none, absolute, percentile
    "artifact_clip_abs_value": None,             # in source_unit, e.g. 150.0 when source_unit=microvolts
    "artifact_clip_percentile": 99.5,

    # ------------------------------------------------------------------
    # MNE Raw-level preprocessing
    # ------------------------------------------------------------------
    "reference_mode": "average",                # average, none
    "reference_timing": "before_resample_filter",  # before_resample_filter, after_resample_before_filter, after_filter
    "resample": True,
    "resample_sfreq": 128,

    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",                     # fir, iir
    "filter_phase": "zero",                     # zero, zero-double, minimum (FIR only)
    "filter_fir_design": "firwin",              # firwin, firwin2 (FIR only)
    "filter_l_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_h_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_iir_params": None,                    # example: {"order": 2, "ftype": "butter"}

    "notch_freqs": None,                          # example: [50.0]
    "notch_before_bandpass": False,

    # ------------------------------------------------------------------
    # Windowing and post-window cleaning
    # ------------------------------------------------------------------
    "target_window_s": 4.2,
    "target_window_samples": 537,
    "mi_window_start_s": 1.5,

    "reject_bad_trials": False,
    "reject_peak_to_peak_threshold": None,       # in final_model_unit
    "reject_abs_threshold": None,                # in final_model_unit
    "min_trials_per_class_after_reject": None,

    # ------------------------------------------------------------------
    # Fold-safe normalization. train_* modes are fit on each training split only.
    # ------------------------------------------------------------------
    "normalization_mode": "none",               # none, train_global_zscore, train_channel_zscore, train_channel_robust, trial_global_zscore, trial_channel_zscore
    "normalization_eps": 1e-6,

    # ------------------------------------------------------------------
    # Model / downstream strategy
    # ------------------------------------------------------------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",                          # new, full
    "warmup_epochs": 10,

    # ------------------------------------------------------------------
    # Evaluation protocol
    # ------------------------------------------------------------------
    "evaluation_mode": "stratified_kfold",      # stratified_kfold, liu2024_repeated_60_40, repeated_stratified_split
    "cv_folds": 5,
    "n_repeats": 10,
    "test_size": 0.4,
    "split_random_state": 2026,
    "assert_balanced_folds": True,

    # ------------------------------------------------------------------
    # Training hyperparameters
    # ------------------------------------------------------------------
    "batch_size": 4,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "val_split": 0.2,
    "learning_rate": 0.0003,
    "optimizer_name": "adam",                   # adam, adamw
    "weight_decay": 0.0,
    "gradient_clip_norm": None,
    "checkpoint_metric": "valid_loss",           # valid_loss, valid_balanced_accuracy
    "label_smoothing": 0.0,
    "prediction_balance_loss_weight": 1.0,

    # ------------------------------------------------------------------
    # braindecode on-the-fly augmentation.
    # Applied to the TRAINING iterator ONLY (via AugmentedDataLoader), so the
    # validation split skorch carves out internally is never augmented -> no leakage.
    # There is no fixed "number of augmented samples": the model sees a freshly
    # augmented view of the train fold every epoch. Control INTENSITY with each
    # transform's "probability" (how often it fires) and its magnitude params.
    # Ready-to-use configs are in the markdown cell just below CONFIG.
    # ------------------------------------------------------------------
    # "augmentation": {
    #     "enabled": False,        # master switch
    #     "name": "none",          # label, saved with artifacts
    #     "random_state": 2026,
    #     # each entry: {"name": <transform>, "probability": 0..1, <transform params>}
    #     "transforms": [],
    # },

    "augmentation": {
        "enabled": True,
        "name": "time_mask",
        "random_state": 2026,
        "transforms": [
        {
            "mask_len_samples": 64,
            "name": "smooth_time_mask",
            "probability": 0.5
        }
        ]
    },

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
    "val_split_random_state": 2026,

    # ------------------------------------------------------------------
    # Diagnostics / interpretation
    # ------------------------------------------------------------------
    "extract_spatial_conv_weights": True,
    "save_spatial_weight_plots": False,
    "plot_individual_spatial_filters": False,
    "max_spatial_filters_to_plot": 8,
    "topomap_dpi": 160,
    "topomap_value_mode": "relative_zscore",    # raw, relative_zscore, relative_percent
    "topomap_cmap": "RdBu_r",
    "collapse_threshold": 0.9,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

# ------------------------------------------------------------------ #
#  Supervised-diagnostics additions (everything above is the         #
#  reference CONFIG, carried verbatim).                              #
# ------------------------------------------------------------------ #
CONFIG["experiment_name"] = "supervised_classifier_diagnostics"
CONFIG["artifact_dir"] = str(WORKING_DIR / "artifacts" / "liu2024-supervised-classifier-diagnostics")
CONFIG["config_note"] = ("Standard supervised baselines (no S-JEPA). Conservative: augmentation off, "
                         "prediction-balance loss penalty off, preprocessing unchanged from reference.")

# --- Deliberate conservative overrides (documented, not silent) ---
# 1) Turn OFF the prediction-balance loss penalty so a collapsing model is visible in the
#    metrics rather than being pushed toward balance by the loss.
CONFIG["prediction_balance_loss_weight"] = 0.0
CONFIG["label_smoothing"] = 0.0
# 2) Disable augmentation by default (configurable). A sanity-check baseline should be plain.
CONFIG["augmentation"] = {"enabled": False, "name": "none", "random_state": 2026, "transforms": []}

# --- Conservative EEGNet supervised-training settings (this notebook's only NN) ---
# The reference's 5000 epochs / lr 3e-4 were tuned for the S-JEPA head; for a compact CNN
# sanity check we use modest, standard settings. All exposed here.
CONFIG["batch_size"] = 16
CONFIG["n_epochs"] = 300
CONFIG["early_stopping_patience"] = 30
CONFIG["learning_rate"] = 0.001
CONFIG["val_split"] = 0.2
CONFIG["checkpoint_metric"] = "valid_loss"

# --- Baseline method registry. enabled=False or a missing dependency -> skipped. ---
CONFIG["methods"] = {
    "logreg_bandpower": {"type": "sklearn_features", "feature": "logvar", "enabled": True},
    "logreg_multiband": {"type": "sklearn_features", "feature": "multiband_logvar",
                          "bands": [[8, 12], [12, 16], [16, 24], [24, 30]], "enabled": False},
    "csp_lda":   {"type": "csp",   "n_components": 4, "enabled": True},
    "fbcsp_lda": {"type": "fbcsp", "n_components": 4,
                   "bands": [[4, 8], [8, 12], [12, 16], [16, 20], [20, 24], [24, 30]], "enabled": True},
    "eegnet":    {"type": "eegnet", "enabled": True},
    "riemann_mdm": {"type": "riemann", "cov_estimator": "oas", "enabled": True},
}

# --- Window-start sensitivity sweep (cheap baseline re-run across start times) ---
CONFIG["window_sensitivity"] = {
    "enabled": True,
    "method": "logreg_bandpower",          # cheap, runs everywhere
    "mi_window_start_grid_s": [0.5, 1.0, 1.5, 2.0, 2.5, 3.0],
    "max_subjects": None,                  # None = all subjects
}

# --- Class-index mapping for one-class / right-hand diagnostics (VERIFY against dataset docs) ---
CONFIG["class_names"] = ["class_0", "class_1"]
CONFIG["positive_class_index"] = 1

# --- Optional: prior S-JEPA run dir to compare against (set to a path string to enable) ---
CONFIG["sjepa_reference_run_dir"] = None

print(f"Experiment: {CONFIG['experiment_name']}")
print(f"Methods:    {[m for m,s in CONFIG['methods'].items() if s.get('enabled')]}")
print("Conservative overrides: prediction_balance_loss_weight=0.0, augmentation disabled.")


Experiment: supervised_classifier_diagnostics
Methods:    ['logreg_bandpower', 'csp_lda', 'fbcsp_lda', 'eegnet', 'riemann_mdm']
Conservative overrides: prediction_balance_loss_weight=0.0, augmentation disabled.


## 2.3 Derived constants (carried verbatim)

In [4]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels.
# Keep the 29 EEG channels used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2

if bool(CONFIG.get("resample", True)):
    EFFECTIVE_SFREQ = float(CONFIG.get("resample_sfreq", 128))
else:
    EFFECTIVE_SFREQ = float(LIU_SOURCE_SFREQ)

CONFIG["effective_sfreq"] = EFFECTIVE_SFREQ
CONFIG["sfreq"] = EFFECTIVE_SFREQ  # compatibility with existing cells/artifacts

if CONFIG.get("target_window_samples", None) is None:
    WINDOW_SAMPLES = int(round(float(CONFIG["target_window_s"]) * EFFECTIVE_SFREQ))
else:
    WINDOW_SAMPLES = int(CONFIG["target_window_samples"])

TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * EFFECTIVE_SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

PREPROCESSING_KEYS = [
    "source_unit", "final_model_unit",
    "demean_mode", "baseline_window_s", "detrend_mode", "eog_correction",
    "artifact_clip_mode", "artifact_clip_abs_value", "artifact_clip_percentile",
    "reference_mode", "reference_timing", "resample", "resample_sfreq", "effective_sfreq",
    "filter_enabled", "filter_low", "filter_high", "filter_method", "filter_phase",
    "filter_fir_design", "filter_l_trans_bandwidth", "filter_h_trans_bandwidth", "filter_iir_params",
    "notch_freqs", "notch_before_bandpass",
    "mi_window_start_s", "target_window_s", "target_window_samples",
    "reject_bad_trials", "reject_peak_to_peak_threshold", "reject_abs_threshold",
    "normalization_mode", "normalization_eps",
]

TRAINING_KEYS = [
    "strategy", "batch_size", "learning_rate", "optimizer_name", "weight_decay",
    "val_split", "early_stopping_patience", "n_epochs",
    "augmentation",
]

EVALUATION_KEYS = [
    "evaluation_mode", "cv_folds", "n_repeats", "test_size",
    "cv_random_state", "split_random_state", "val_split_random_state",
]

def summarize_selected_config(keys):
    return {k: CONFIG.get(k) for k in keys}

PREPROCESSING_CONFIG = summarize_selected_config(PREPROCESSING_KEYS)
TRAINING_CONFIG = summarize_selected_config(TRAINING_KEYS)
EVALUATION_CONFIG = summarize_selected_config(EVALUATION_KEYS)

def print_config_block(title, values):
    print(title)
    for key, value in values.items():
        print(f"  {key:34s}: {value}")

print("Effective Liu2024 Source MAT settings:")
print(f"  Experiment:                        {CONFIG.get('experiment_name')}")
print(f"  Note:                              {CONFIG.get('config_note')}")
print(f"  Channels:                          {len(EEG_CHANNEL_NAMES)}")
print(f"  Channel names:                     {EEG_CHANNEL_NAMES}")
print(f"  Source sfreq:                      {LIU_SOURCE_SFREQ} Hz")
print(f"  Effective sfreq:                   {EFFECTIVE_SFREQ} Hz")
print(f"  MI window start / samples:         {CONFIG['mi_window_start_s']} s / {WINDOW_SAMPLES}")
print(f"  Effective window duration:         {TARGET_TRIAL_DURATION_S:.4f} s")
print(f"  Evaluation mode:                   {CONFIG.get('evaluation_mode')}")
print(f"  Fixed seed:                        base={CONFIG.get('seed')} | cv={CONFIG.get('cv_random_state')} | split={CONFIG.get('split_random_state')} | val={CONFIG.get('val_split_random_state')}")
print_config_block("\nPreprocessing config:", PREPROCESSING_CONFIG)
print_config_block("\nTraining config:", TRAINING_CONFIG)
print_config_block("\nEvaluation config:", EVALUATION_CONFIG)


Effective Liu2024 Source MAT settings:
  Experiment:                        supervised_classifier_diagnostics
  Note:                              Standard supervised baselines (no S-JEPA). Conservative: augmentation off, prediction-balance loss penalty off, preprocessing unchanged from reference.
  Channels:                          29
  Channel names:                     ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']
  Source sfreq:                      500 Hz
  Effective sfreq:                   128.0 Hz
  MI window start / samples:         1.5 s / 537
  Effective window duration:         4.1953 s
  Evaluation mode:                   stratified_kfold
  Fixed seed:                        base=2026 | cv=2026 | split=2026 | val=2026

Preprocessing config:
  source_unit                       : microvolts
  final_model_unit                  : microvol

## 2.4 Artifacts and logging (carried verbatim)

In [5]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


[2026-06-13 09:48:39] Run ID:     20260613_0948_d81691de
[2026-06-13 09:48:39] Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-supervised-classifier-diagnostics/20260613_0948_d81691de
[2026-06-13 09:48:39] Config:     /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-supervised-classifier-diagnostics/20260613_0948_d81691de/config.json


## 2.5 Reproducibility (carried verbatim)

In [6]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


[2026-06-13 09:48:40] Using device: mps
[2026-06-13 09:48:40] Seed initialized: 2026


# 3. Carried machinery (verbatim)

Data loading, preprocessing, dataset classes, JSON-safe helpers, the augmentation-transform
builder + fold-safe normalizer + classifier + metrics/collapse/probability diagnostics, and
the evaluation-split builder — all unchanged from the reference. The S-JEPA model, the
pretrained-checkpoint logic, and the S-JEPA training runner are **not** carried.

### 3.1 Data-loading helpers

In [7]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


### 3.2 Preprocessing pipeline

In [8]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES),  # type: ignore
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def source_values_to_mne_volts(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("source_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e-6
    if unit in ("mv", "millivolt", "millivolts"):
        return arr * 1e-3
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported source_unit={config.get('source_unit')}")

def mne_volts_to_model_unit(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("final_model_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e6
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported final_model_unit={config.get('final_model_unit')}")

def _none_like(value):
    return value is None or str(value).lower() in ("none", "off", "false", "")

def build_preprocessing_pipeline(config):
    """Return a readable pipeline plan.

    The pipeline is represented as a list of dictionaries rather than hidden global logic.
    Every row is logged and saved in the run metadata through the preprocessing step list.
    """
    pipeline = []

    # Fixed source structure.
    pipeline.append({
        "stage": "source",
        "name": "select_eeg_channels",
        "description": "select Liu EEG channels, drop CPz reference, EOG, and marker before model input",
        "enabled": True,
    })

    pipeline.append({
        "stage": "source",
        "name": "demean",
        "mode": config.get("demean_mode", "none"),
        "baseline_window_s": config.get("baseline_window_s"),
        "enabled": not _none_like(config.get("demean_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "detrend",
        "mode": config.get("detrend_mode", "none"),
        "enabled": not _none_like(config.get("detrend_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "eog_correction",
        "mode": config.get("eog_correction", "none"),
        "enabled": not _none_like(config.get("eog_correction", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "artifact_clipping",
        "mode": config.get("artifact_clip_mode", "none"),
        "abs_value": config.get("artifact_clip_abs_value"),
        "percentile": config.get("artifact_clip_percentile"),
        "enabled": not _none_like(config.get("artifact_clip_mode", "none")),
    })

    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()
    if reference_timing == "before_resample_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "before_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({"stage": "mne_raw", "name": "resample", "sfreq": config.get("resample_sfreq"), "enabled": bool(config.get("resample", True))})

    if reference_timing == "after_resample_before_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    pipeline.append({
        "stage": "mne_raw",
        "name": "bandpass_filter",
        "enabled": bool(config.get("filter_enabled", True)),
        "l_freq": config.get("filter_low"),
        "h_freq": config.get("filter_high"),
        "method": config.get("filter_method"),
        "phase": config.get("filter_phase"),
        "fir_design": config.get("filter_fir_design"),
        "iir_params": config.get("filter_iir_params"),
    })

    if reference_timing == "after_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if not bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "after_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({
        "stage": "window",
        "name": "crop_fixed_mi_window",
        "start_s": config.get("mi_window_start_s"),
        "target_window_samples": config.get("target_window_samples"),
        "enabled": True,
    })

    pipeline.append({
        "stage": "window",
        "name": "bad_trial_rejection",
        "enabled": bool(config.get("reject_bad_trials", False)),
        "peak_to_peak_threshold": config.get("reject_peak_to_peak_threshold"),
        "abs_threshold": config.get("reject_abs_threshold"),
    })

    pipeline.append({
        "stage": "split",
        "name": "fold_safe_normalization",
        "mode": config.get("normalization_mode", "none"),
        "enabled": not _none_like(config.get("normalization_mode", "none")),
    })

    return pipeline

def describe_pipeline(pipeline):
    lines = []
    for step in pipeline:
        status = "ON" if step.get("enabled", False) else "off"
        parts = [f"[{status}] {step.get('stage')}::{step.get('name')}"]
        for key, value in step.items():
            if key not in ("stage", "name", "description", "enabled") and value is not None:
                parts.append(f"{key}={value}")
        if step.get("description"):
            parts.append(f"- {step['description']}")
        lines.append(" | ".join(parts))
    return lines

PREPROCESSING_PIPELINE = build_preprocessing_pipeline(CONFIG)
print("Configured preprocessing pipeline:")
for line in describe_pipeline(PREPROCESSING_PIPELINE):
    print("  - " + line)

def apply_source_demean(X_eeg, subject_id, config, steps):
    mode = str(config.get("demean_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source demean skipped")
        return X

    if mode == "trial_mean":
        steps.append("source demean: subtract trial/channel mean over time")
        return X - X.mean(axis=-1, keepdims=True)

    if mode == "baseline_window_mean":
        baseline = config.get("baseline_window_s", [0.0, 2.0])
        if baseline is None or len(baseline) != 2:
            raise ValueError("baseline_window_s must be [start_s, stop_s] for baseline_window_mean.")
        start_s, stop_s = float(baseline[0]), float(baseline[1])
        start = int(round(start_s * LIU_SOURCE_SFREQ))
        stop = int(round(stop_s * LIU_SOURCE_SFREQ))
        if start < 0 or stop <= start or stop > X.shape[-1]:
            raise ValueError(f"Subject {subject_id}: invalid baseline_window_s={baseline} for source length {X.shape[-1]}")
        steps.append(f"source demean: subtract baseline mean {baseline}s")
        return X - X[:, :, start:stop].mean(axis=-1, keepdims=True)

    raise ValueError(f"Unsupported demean_mode={config.get('demean_mode')}")

def apply_source_detrend(X_eeg, subject_id, config, steps):
    mode = str(config.get("detrend_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)
    if mode in ("none", "off", "false"):
        steps.append("source detrend skipped")
        return X
    if mode == "constant":
        steps.append("source detrend: scipy.signal.detrend(type='constant')")
        return signal.detrend(X, axis=-1, type="constant")
    if mode == "linear":
        steps.append("source detrend: scipy.signal.detrend(type='linear')")
        return signal.detrend(X, axis=-1, type="linear")
    raise ValueError(f"Unsupported detrend_mode={config.get('detrend_mode')}")

def apply_eog_correction(X_eeg, rawdata, subject_id, config, steps):
    mode = str(config.get("eog_correction", "none")).lower()
    if mode in ("none", "off", "false"):
        steps.append("EOG correction skipped")
        return X_eeg

    if mode != "linear_regression":
        raise ValueError(
            "Only eog_correction='linear_regression' is implemented in this source-MAT notebook. "
            "ICA is intentionally not included because the source pipeline drops EOG before model input and "
            "the dataset has only 40 trials per subject."
        )

    if rawdata.shape[1] <= max(SOURCE_EOG_CHANNEL_INDICES):
        raise ValueError(f"Subject {subject_id}: rawdata does not contain expected EOG channels.")

    X = np.asarray(X_eeg, dtype=np.float64)
    eog = np.asarray(rawdata[:, SOURCE_EOG_CHANNEL_INDICES, :], dtype=np.float64)

    n_trials, n_chans, n_samples = X.shape
    eog_2d = eog.transpose(0, 2, 1).reshape(-1, len(SOURCE_EOG_CHANNEL_INDICES))
    eeg_2d = X.transpose(0, 2, 1).reshape(-1, n_chans)

    design = np.column_stack([np.ones(eog_2d.shape[0]), eog_2d])
    beta, *_ = np.linalg.lstsq(design, eeg_2d, rcond=None)
    eog_contribution = design[:, 1:] @ beta[1:, :]
    corrected = eeg_2d - eog_contribution
    corrected = corrected.reshape(n_trials, n_samples, n_chans).transpose(0, 2, 1)

    steps.append("EOG correction: linear regression using HEOG/VEOG before dropping EOG")
    return corrected

def apply_source_artifact_clipping(X_eeg, subject_id, config, steps):
    mode = str(config.get("artifact_clip_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source artifact clipping skipped")
        return X, {"artifact_clip_applied": False, "artifact_clip_threshold": None}

    if mode == "absolute":
        threshold = config.get("artifact_clip_abs_value")
        if threshold is None:
            raise ValueError("artifact_clip_abs_value must be set when artifact_clip_mode='absolute'.")
        threshold = float(threshold)
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: absolute ±{threshold:g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    if mode == "percentile":
        pct = float(config.get("artifact_clip_percentile", 99.5))
        threshold = float(np.nanpercentile(np.abs(X), pct))
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: percentile {pct:g}% -> ±{threshold:.4g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_percentile": pct,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    raise ValueError(f"Unsupported artifact_clip_mode={config.get('artifact_clip_mode')}")

def apply_reference(raw, config, steps, timing_label):
    mode = str(config.get("reference_mode", "average")).lower()
    if mode in ("none", "off", "false"):
        steps.append(f"reference skipped at {timing_label}")
        return raw
    if mode == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
        steps.append(f"average reference at {timing_label}")
        return raw
    raise ValueError(f"Unsupported reference_mode={config.get('reference_mode')}")

def apply_notch(raw, config, steps, timing_label):
    freqs = config.get("notch_freqs", None)
    if freqs is None or freqs == []:
        return raw
    raw.notch_filter(freqs=freqs, verbose=False)
    steps.append(f"notch filter {freqs} Hz at {timing_label}")
    return raw

def apply_resample(raw, config, steps):
    if bool(config.get("resample", True)):
        target = float(config.get("resample_sfreq", 128))
        raw.resample(target, verbose=False)
        steps.append(f"resample to {target:g} Hz")
    else:
        steps.append("resample skipped; kept source 500 Hz")
    return raw

def _float_or_none_or_auto(value):
    if value is None:
        return None
    if isinstance(value, str) and value.lower() == "auto":
        return "auto"
    return float(value)

def apply_bandpass(raw, config, steps):
    if not bool(config.get("filter_enabled", True)):
        steps.append("bandpass skipped")
        return raw

    l_freq = config.get("filter_low", None)
    h_freq = config.get("filter_high", None)
    l_freq = None if l_freq is None else float(l_freq)
    h_freq = None if h_freq is None else float(h_freq)

    method = str(config.get("filter_method", "fir")).lower()
    if method == "iir":
        iir_params = config.get("filter_iir_params", None)
        if iir_params is None:
            iir_params = {"order": 2, "ftype": "butter"}
        raw.filter(l_freq=l_freq, h_freq=h_freq, method="iir", iir_params=iir_params, verbose=False)
        steps.append(f"IIR bandpass {l_freq}–{h_freq} Hz | params={iir_params}")
    elif method == "fir":
        raw.filter(
            l_freq=l_freq,
            h_freq=h_freq,
            method="fir",
            phase=config.get("filter_phase", "zero"),
            fir_design=config.get("filter_fir_design", "firwin"),
            l_trans_bandwidth=_float_or_none_or_auto(config.get("filter_l_trans_bandwidth", "auto")),
            h_trans_bandwidth=_float_or_none_or_auto(config.get("filter_h_trans_bandwidth", "auto")),
            verbose=False,
        )
        steps.append(
            f"FIR bandpass {l_freq}–{h_freq} Hz | phase={config.get('filter_phase')} | "
            f"fir_design={config.get('filter_fir_design')}"
        )
    else:
        raise ValueError(f"Unsupported filter_method={config.get('filter_method')}")
    return raw

def apply_mne_raw_pipeline(raw, config, steps):
    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()

    if reference_timing == "before_resample_filter":
        raw = apply_reference(raw, config, steps, "before resample/filter")

    if bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "before resample/filter")

    raw = apply_resample(raw, config, steps)

    if reference_timing == "after_resample_before_filter":
        raw = apply_reference(raw, config, steps, "after resample before filter")

    raw = apply_bandpass(raw, config, steps)

    if reference_timing == "after_filter":
        raw = apply_reference(raw, config, steps, "after filter")

    if not bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "after bandpass")

    return raw

def maybe_reject_bad_trials(X_win, y, subject_id, config, steps):
    stats = {
        "reject_bad_trials": bool(config.get("reject_bad_trials", False)),
        "n_trials_before_reject": int(len(y)),
        "n_trials_after_reject": int(len(y)),
        "n_rejected_trials": 0,
        "rejection_skipped": False,
    }

    if not bool(config.get("reject_bad_trials", False)):
        steps.append("bad-trial rejection skipped")
        return X_win, y, stats

    keep = np.ones(len(y), dtype=bool)

    ptp_threshold = config.get("reject_peak_to_peak_threshold", None)
    if ptp_threshold is not None:
        ptp = np.ptp(X_win, axis=-1).max(axis=1)
        keep &= ptp <= float(ptp_threshold)
        stats["reject_peak_to_peak_threshold"] = float(ptp_threshold)
        stats["max_trial_peak_to_peak"] = float(np.max(ptp))

    abs_threshold = config.get("reject_abs_threshold", None)
    if abs_threshold is not None:
        max_abs = np.max(np.abs(X_win), axis=(1, 2))
        keep &= max_abs <= float(abs_threshold)
        stats["reject_abs_threshold"] = float(abs_threshold)
        stats["max_trial_abs"] = float(np.max(max_abs))

    proposed_y = y[keep]
    min_required = config.get("min_trials_per_class_after_reject", None)
    if min_required is None:
        if config.get("evaluation_mode") == "liu2024_repeated_60_40":
            min_required = 12
        else:
            min_required = int(config.get("cv_folds", 5))
    proposed_counts = np.bincount(proposed_y, minlength=TARGET_N_CLASSES)

    if len(proposed_y) == 0 or proposed_counts.min() < int(min_required):
        steps.append(
            "bad-trial rejection skipped because it would leave too few samples "
            f"per class: proposed_counts={proposed_counts.tolist()}, min_required={min_required}"
        )
        stats["rejection_skipped"] = True
        stats["proposed_class_counts_after_reject"] = proposed_counts.tolist()
        return X_win, y, stats

    X_new = X_win[keep]
    y_new = proposed_y
    stats["n_trials_after_reject"] = int(len(y_new))
    stats["n_rejected_trials"] = int(np.sum(~keep))
    stats["class_counts_after_reject"] = np.bincount(y_new, minlength=TARGET_N_CLASSES).tolist()
    steps.append(
        f"bad-trial rejection applied: rejected={stats['n_rejected_trials']} / {stats['n_trials_before_reject']} | "
        f"class_counts={stats['class_counts_after_reject']}"
    )
    return X_new, y_new, stats

def preprocess_subject_configurable(rawdata, labels, subject_id, config=None):
    """Apply the configured preprocessing pipeline to one Liu2024 source subject.

    Order:
      1. source-domain operations: channel selection, mean removal, detrending, EOG regression, clipping
      2. MNE RawArray operations: reference, notch, resample, bandpass
      3. window crop and optional trial rejection
      4. fold-safe normalization later inside training split code
    """
    config = CONFIG if config is None else config

    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")

    n_trials = rawdata.shape[0]
    steps = describe_pipeline(build_preprocessing_pipeline(config))
    runtime_steps = []
    preprocessing_stats = {}

    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    runtime_steps.append("select 29 EEG channels; drop CPz source reference, EOG, and marker")

    X_eeg = apply_source_demean(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_source_detrend(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_eog_correction(X_eeg, rawdata, subject_id, config, runtime_steps)
    X_eeg, clip_stats = apply_source_artifact_clipping(X_eeg, subject_id, config, runtime_steps)
    preprocessing_stats.update(clip_stats)

    X_eeg_volts = source_values_to_mne_volts(X_eeg, config)
    runtime_steps.append(f"convert source {config.get('source_unit')} to MNE volts")

    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(LIU_SOURCE_SFREQ)
    raw = mne.io.RawArray(continuous, info, verbose=False)

    raw = apply_mne_raw_pipeline(raw, config, runtime_steps)

    effective_sfreq = float(raw.info["sfreq"])
    data = mne_volts_to_model_unit(raw.get_data(), config)
    runtime_steps.append(f"convert MNE volts to model {config.get('final_model_unit')}")

    expected_samples_per_trial = int(round(rawdata.shape[2] * effective_sfreq / float(LIU_SOURCE_SFREQ)))
    total_expected = n_trials * expected_samples_per_trial
    if data.shape[1] != total_expected:  # type: ignore
        n_full = data.shape[1] // n_trials  # type: ignore
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]  # type: ignore
        runtime_steps.append(f"trim continuous samples to full trials: {expected_samples_per_trial} samples/trial")

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)  # type: ignore

    start_sample = int(round(float(config["mi_window_start_s"]) * effective_sfreq))
    window_samples = int(config["target_window_samples"]) if config.get("target_window_samples") is not None else int(round(float(config["target_window_s"]) * effective_sfreq))
    stop_sample = start_sample + window_samples

    if stop_sample > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop [{start_sample}:{stop_sample}] exceeds trial length "
            f"{X_rs.shape[-1]} at effective_sfreq={effective_sfreq}"
        )

    X_win = X_rs[:, :, start_sample:stop_sample]
    runtime_steps.append(f"crop fixed window samples [{start_sample}:{stop_sample}]")

    y = labels_to_zero_based(labels)
    X_win, y, reject_stats = maybe_reject_bad_trials(X_win, y, subject_id, config, runtime_steps)
    preprocessing_stats.update(reject_stats)

    # Save both the intended pipeline and the actual runtime steps.
    preprocessing_stats["pipeline_plan"] = steps
    preprocessing_stats["runtime_steps"] = runtime_steps

    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial), runtime_steps, preprocessing_stats


[2026-06-13 09:48:40] Configured preprocessing pipeline:
[2026-06-13 09:48:40]   - [ON] source::select_eeg_channels | - select Liu EEG channels, drop CPz reference, EOG, and marker before model input
[2026-06-13 09:48:40]   - [off] source::demean | mode=none | baseline_window_s=[0.0, 2.0]
[2026-06-13 09:48:40]   - [off] source::detrend | mode=none
[2026-06-13 09:48:40]   - [off] source::eog_correction | mode=none
[2026-06-13 09:48:40]   - [off] source::artifact_clipping | mode=none | percentile=99.5
[2026-06-13 09:48:40]   - [ON] mne_raw::reference | timing=before_resample_filter | mode=average
[2026-06-13 09:48:40]   - [ON] mne_raw::resample | sfreq=128
[2026-06-13 09:48:40]   - [ON] mne_raw::bandpass_filter | l_freq=0.5 | h_freq=40.0 | method=fir | phase=zero | fir_design=firwin
[2026-06-13 09:48:40]   - [off] mne_raw::notch_filter | timing=after_bandpass
[2026-06-13 09:48:40]   - [ON] window::crop_fixed_mi_window | start_s=1.5 | target_window_samples=537
[2026-06-13 09:48:40]   - [o

### 3.3 Dataset classes

In [9]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)
        if self.X.ndim != 3:
            raise ValueError(f"SubjectArrayDataset expects X as N x C x T, got shape={self.X.shape}.")
        if len(self.X) != len(self.y):
            raise ValueError(f"X/y length mismatch: {len(self.X)} windows vs {len(self.y)} labels.")

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected one EEG window as C x T, got shape={x.shape}.")
        return x, int(self.y[idx])


class FoldNormalizedDataset(Dataset):
    """Wrap a dataset and apply either fold-fitted or trial-wise normalization."""

    def __init__(self, dataset, normalizer_state):
        self.dataset = dataset
        self.normalizer_state = normalizer_state or {"mode": "none"}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = apply_normalizer_to_array(x, self.normalizer_state)
        if x.ndim != 2:
            raise ValueError(f"Normalization must return C x T, got shape={x.shape}.")
        return x.astype(np.float32), int(y)


### 3.4 JSON-safe + spatial-conv helpers

In [10]:
def _json_safe_float(value, decimals=8):
    if value is None:
        return None
    value = float(np.nan_to_num(value, nan=0.0, posinf=0.0, neginf=0.0))
    return round(value, decimals)

def _json_safe_float_list(values, decimals=8):
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.round(arr, decimals=decimals).tolist()

def get_model_module(model_or_clf):
    return model_or_clf.module_ if hasattr(model_or_clf, "module_") else model_or_clf

def _find_first_spatial_weight(model_or_clf):
    model = get_model_module(model_or_clf)
    if not hasattr(model, "spatial_conv"):
        return None, None

    candidates = []
    for name, param in model.named_parameters():
        if name.startswith("spatial_conv.") and name.endswith("weight") and param.ndim >= 2:
            candidates.append((name, param.detach().cpu().clone()))
    if candidates:
        candidates = sorted(candidates, key=lambda x: (0 if "spatial_conv.1.weight" in x[0] else 1, x[0]))
        return candidates[0]

    for name, module in model.spatial_conv.named_modules():
        if hasattr(module, "weight") and module.weight is not None and module.weight.ndim >= 2:
            return f"spatial_conv.{name}.weight", module.weight.detach().cpu().clone()
    return None, None

def _spatial_weight_to_channel_matrix(weight_tensor, n_chans):
    w = weight_tensor.detach().cpu().float().numpy()
    mat = w.copy() if w.ndim == 2 else w.reshape(w.shape[0], -1)
    if mat.shape[1] == n_chans:
        pass
    elif mat.shape[0] == n_chans:
        mat = mat.T
    elif mat.shape[1] % n_chans == 0:
        mat = mat.reshape(mat.shape[0], -1, n_chans).mean(axis=1)
    elif mat.size % n_chans == 0:
        mat = mat.reshape(-1, n_chans)
    else:
        raise ValueError(
            f"Cannot reshape spatial_conv weight of shape {tuple(w.shape)} into channel matrix with n_chans={n_chans}."
        )
    return mat

def get_spatial_conv_weight_matrix(model_or_clf, ch_names):
    param_name, weight_tensor = _find_first_spatial_weight(model_or_clf)
    if weight_tensor is None:
        raise RuntimeError("Could not find spatial_conv weight on model.")
    weight_matrix = _spatial_weight_to_channel_matrix(weight_tensor, n_chans=len(ch_names))
    return weight_matrix.copy(), param_name

def compute_spatial_update_stats(initial_weight_matrix, final_weight_matrix, parameter_name):
    if initial_weight_matrix is None or final_weight_matrix is None:
        return None
    w0 = np.asarray(initial_weight_matrix, dtype=float)
    w1 = np.asarray(final_weight_matrix, dtype=float)
    if w0.shape != w1.shape:
        return {
            "available": False,
            "parameter_name": str(parameter_name),
            "reason": f"Initial/final spatial weights have different shapes: {w0.shape} vs {w1.shape}",
        }
    delta = w1 - w0
    init_l2 = float(np.linalg.norm(w0))
    final_l2 = float(np.linalg.norm(w1))
    delta_l2 = float(np.linalg.norm(delta))
    max_abs_delta = float(np.max(np.abs(delta))) if delta.size else 0.0
    relative_delta = delta_l2 / (init_l2 + 1e-12)
    return {
        "available": True,
        "parameter_name": str(parameter_name),
        "initial_shape": list(w0.shape),
        "final_shape": list(w1.shape),
        "initial_l2": _json_safe_float(init_l2),
        "final_l2": _json_safe_float(final_l2),
        "delta_l2": _json_safe_float(delta_l2),
        "delta_max_abs": _json_safe_float(max_abs_delta),
        "relative_delta": _json_safe_float(relative_delta),
        "changed": bool(delta_l2 > 1e-10),
    }

def extract_spatial_conv_summary(model_or_clf, ch_names):
    param_name, weight_tensor = _find_first_spatial_weight(model_or_clf)
    if weight_tensor is None:
        return {"available": False, "reason": "No spatial_conv weight found on model."}

    try:
        weight_matrix = _spatial_weight_to_channel_matrix(weight_tensor, n_chans=len(ch_names))
    except Exception as exc:
        return {
            "available": False,
            "reason": str(exc),
            "parameter_name": param_name,
            "raw_weight_shape": list(weight_tensor.shape),
        }

    denom = np.max(np.abs(weight_matrix), axis=1, keepdims=True)
    denom[denom == 0] = 1.0
    weight_matrix_norm = weight_matrix / denom

    channel_abs_mean = np.mean(np.abs(weight_matrix), axis=0)
    channel_signed_mean = np.mean(weight_matrix, axis=0)
    channel_l2 = np.sqrt(np.mean(weight_matrix ** 2, axis=0))

    top_order = np.argsort(channel_abs_mean)[::-1]
    top_channels_by_abs_weight = [
        {
            "rank": int(rank + 1),
            "channel_index": int(ch_idx),
            "channel_name": str(ch_names[ch_idx]),
            "abs_mean_weight": _json_safe_float(channel_abs_mean[ch_idx]),
            "signed_mean_weight": _json_safe_float(channel_signed_mean[ch_idx]),
            "l2_weight": _json_safe_float(channel_l2[ch_idx]),
        }
        for rank, ch_idx in enumerate(top_order[:min(15, len(top_order))])
    ]

    return {
        "available": True,
        "parameter_name": param_name,
        "raw_weight_shape": list(weight_tensor.shape),
        "weight_matrix_shape": list(weight_matrix.shape),
        "n_virtual_filters": int(weight_matrix.shape[0]),
        "n_input_channels": int(weight_matrix.shape[1]),
        "channel_names": list(ch_names),
        "weight_matrix": _json_safe_float_list(weight_matrix),
        "weight_matrix_norm": _json_safe_float_list(weight_matrix_norm),
        "channel_abs_mean": _json_safe_float_list(channel_abs_mean),
        "channel_signed_mean": _json_safe_float_list(channel_signed_mean),
        "channel_l2": _json_safe_float_list(channel_l2),
        "top_channels_by_abs_weight": top_channels_by_abs_weight,
    }

def summarize_spatial_conv_for_log(spatial_summary, top_k=8):
    if spatial_summary is None:
        return "spatial_conv=N/A"
    if not spatial_summary.get("available", False):
        return f"spatial_conv=unavailable ({spatial_summary.get('reason')})"
    ch_names = spatial_summary["channel_names"]
    scores = np.asarray(spatial_summary["channel_abs_mean"], dtype=float)
    order = np.argsort(scores)[::-1][:min(top_k, len(scores))]
    top_channels = [(ch_names[i], float(scores[i])) for i in order]
    return f"spatial_conv={spatial_summary['weight_matrix_shape']} top_abs_channels={top_channels}"

def summarize_spatial_update_for_log(update_stats):
    if not update_stats:
        return "spatial_update=N/A"
    if not update_stats.get("available", False):
        return f"spatial_update=unavailable ({update_stats.get('reason')})"
    return (
        "spatial_update="
        f"delta_l2={update_stats['delta_l2']:.8f} "
        f"max_abs={update_stats['delta_max_abs']:.8f} "
        f"relative={update_stats['relative_delta']:.8f} "
        f"changed={update_stats['changed']}"
    )

def _make_eeg_info_for_topomap(ch_names):
    try:
        info = mne.pick_info(EEG_INFO.copy(), mne.pick_types(EEG_INFO, eeg=True, meg=False, stim=False))
        picks = [info.ch_names.index(ch) for ch in ch_names if ch in info.ch_names]
        if len(picks) == len(ch_names):
            return mne.pick_info(info, picks)
    except Exception:
        pass

    info = mne.create_info(ch_names=ch_names, sfreq=float(CONFIG["sfreq"]), ch_types="eeg")
    for montage_name in ("standard_1020", "standard_1005"):
        try:
            montage = mne.channels.make_standard_montage(montage_name)
            info.set_montage(montage, match_case=False, on_missing="ignore")
            return info
        except Exception:
            continue
    return info

def transform_topomap_values(values, mode):
    values = np.asarray(values, dtype=float)
    if mode == "raw":
        return values, "mean absolute spatial_conv weight"
    if mode == "relative_zscore":
        std = float(np.nanstd(values))
        if std < 1e-12:
            return np.zeros_like(values), "within-run z-score of mean absolute spatial_conv weight"
        return (values - float(np.nanmean(values))) / std, "within-run z-score of mean absolute spatial_conv weight"
    if mode == "relative_percent":
        mean = float(np.nanmean(values))
        if abs(mean) < 1e-12:
            return np.zeros_like(values), "% deviation from mean absolute spatial_conv weight"
        return 100.0 * (values - mean) / mean, "% deviation from mean absolute spatial_conv weight"
    raise ValueError(f"Unsupported topomap value mode: {mode}")

def _safe_plot_topomap(values, info, axes, cmap="RdBu_r", vlim=None, names=None, contours=0):
    base_kwargs = {
        "data": values,
        "pos": info,
        "axes": axes,
        "show": False,
        "cmap": cmap,
        "contours": contours,
    }
    if names is not None:
        base_kwargs["names"] = names
    if vlim is not None:
        base_kwargs["vlim"] = vlim

    attempts = [dict(base_kwargs)]
    no_names = dict(base_kwargs)
    no_names.pop("names", None)
    attempts.append(no_names)

    if vlim is not None:
        vmin, vmax = vlim
        for kwargs in list(attempts):
            old_kwargs = dict(kwargs)
            old_kwargs.pop("vlim", None)
            old_kwargs["vmin"] = vmin
            old_kwargs["vmax"] = vmax
            attempts.append(old_kwargs)

    last_exc = None
    for kwargs in attempts:
        try:
            return mne.viz.plot_topomap(**kwargs)
        except TypeError as exc:
            last_exc = exc
            continue
    raise last_exc # type: ignore

def _extract_topomap_image(plot_result):
    if isinstance(plot_result, tuple) and len(plot_result) > 0:
        return plot_result[0]
    return plot_result

def plot_spatial_topomap(values, ch_names, title, out_path, value_mode=None, cmap=None):
    values = np.asarray(values, dtype=float)
    mode = value_mode or CONFIG.get("topomap_value_mode", "relative_zscore")
    cmap = cmap or CONFIG.get("topomap_cmap", "RdBu_r")
    plot_values, cbar_label = transform_topomap_values(values, mode)
    info = _make_eeg_info_for_topomap(ch_names)
    try:
        fig, ax = plt.subplots(figsize=(5.8, 5.0))
        vmax = float(np.nanmax(np.abs(plot_values))) if np.any(np.isfinite(plot_values)) else 1.0
        if not np.isfinite(vmax) or vmax < 1e-12:
            vmax = 1.0
        result = _safe_plot_topomap(
            plot_values,
            info,
            axes=ax,
            cmap=cmap,
            vlim=(-vmax, vmax),
            names=ch_names,
            contours=0,
        )
        im = _extract_topomap_image(result)
        if im is not None:
            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label(cbar_label)
        ax.set_title(f"{title}\n{mode}")
        fig.tight_layout()
        fig.savefig(out_path, dpi=int(CONFIG.get("topomap_dpi", 160)), bbox_inches="tight")
        plt.close(fig)
        return True
    except Exception as exc:
        print(f"WARNING: Could not save topomap {out_path}: {exc}")
        plt.close("all")
        return False

def plot_spatial_filter_grid(weight_matrix, ch_names, title, out_path, max_filters=8):
    weight_matrix = np.asarray(weight_matrix, dtype=float)
    n_filters = min(int(max_filters), weight_matrix.shape[0])
    if n_filters <= 0:
        return False

    info = _make_eeg_info_for_topomap(ch_names)
    n_cols = min(4, n_filters)
    n_rows = int(math.ceil(n_filters / n_cols))
    try:
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 3.6 * n_rows))
        axes = np.asarray(axes).reshape(-1)
        vmax = float(np.nanmax(np.abs(weight_matrix[:n_filters]))) if np.any(np.isfinite(weight_matrix[:n_filters])) else 1.0
        if not np.isfinite(vmax) or vmax < 1e-12:
            vmax = 1.0
        last_im = None
        for i in range(n_filters):
            ax = axes[i]
            result = _safe_plot_topomap(
                weight_matrix[i],
                info,
                axes=ax,
                cmap="RdBu_r",
                vlim=(-vmax, vmax),
                contours=0,
            )
            last_im = _extract_topomap_image(result)
            ax.set_title(f"virtual filter {i}")
        for j in range(n_filters, len(axes)):
            axes[j].axis("off")
        if last_im is not None:
            cbar = fig.colorbar(last_im, ax=axes[:n_filters], fraction=0.025, pad=0.04)
            cbar.set_label("normalized signed spatial_conv weight")
        fig.suptitle(title)
        fig.tight_layout()
        fig.savefig(out_path, dpi=int(CONFIG.get("topomap_dpi", 160)), bbox_inches="tight")
        plt.close(fig)
        return True
    except Exception as exc:
        print(f"WARNING: Could not save filter grid {out_path}: {exc}")
        plt.close("all")
        return False


### 3.5 Augmentation transforms (disabled by default via CONFIG)

In [11]:
# ------------------------------------------------------------------
# braindecode augmentation registry + builder.
# Transforms are applied on-the-fly to TRAINING batches only (AugmentedDataLoader),
# so skorch's internal validation split is never augmented (leak-safe).
#
# NOTE: transform constructor parameter names can vary slightly across braindecode
# versions. If a param errors, check `help(<Transform>)` for your installed version.
# ------------------------------------------------------------------
def build_braindecode_transforms(aug_cfg):
    aug_cfg = aug_cfg or {}
    if not aug_cfg.get("enabled", False):
        return []
    specs = aug_cfg.get("transforms", []) or []
    if not specs:
        return []

    try:
        from braindecode.augmentation import (
            GaussianNoise, SmoothTimeMask, TimeReverse, FrequencyShift,
            BandstopFilter, FTSurrogate, ChannelsDropout, ChannelsShuffle, ChannelsSymmetry,
        )
    except Exception as exc:
        raise ImportError(
            "braindecode.augmentation transforms are unavailable. "
            "Upgrade braindecode (pip install -U braindecode)."
        ) from exc

    sfreq = float(CONFIG["resample_sfreq"]) if CONFIG.get("resample", True) else 500.0
    ch_names = list(CH_NAMES)
    rs = int(aug_cfg.get("random_state", CONFIG.get("seed", 2026)))

    transforms = []
    for spec in specs:
        name = str(spec["name"]).lower()
        p = float(spec.get("probability", 0.5))
        params = {k: v for k, v in spec.items() if k not in ("name", "probability")}

        if name == "gaussian_noise":
            # NOTE: std is in the SAME units as the data. With normalization_mode="none"
            # the data is microvolts, so std=0.2 is tiny -> either enable a *_zscore
            # normalization_mode or raise std (e.g. 2-10) for a visible effect.
            transforms.append(GaussianNoise(probability=p, std=float(params.get("std", 0.2)), random_state=rs))
        elif name == "smooth_time_mask":
            transforms.append(SmoothTimeMask(probability=p, mask_len_samples=int(params.get("mask_len_samples", 64)), random_state=rs))
        elif name == "time_reverse":
            transforms.append(TimeReverse(probability=p, random_state=rs))
        elif name == "frequency_shift":
            transforms.append(FrequencyShift(probability=p, sfreq=sfreq, max_delta_freq=float(params.get("max_delta_freq", 1.0)), random_state=rs))
        elif name in ("bandstop_filter", "band_filter"):
            transforms.append(BandstopFilter(probability=p, sfreq=sfreq, bandwidth=float(params.get("bandwidth", 2.0)), max_freq=params.get("max_freq", None), random_state=rs))
        elif name in ("ft_surrogate", "ftsurrogate"):
            transforms.append(FTSurrogate(probability=p, phase_noise_magnitude=float(params.get("phase_noise_magnitude", 1.0)), random_state=rs))
        elif name == "channels_dropout":
            transforms.append(ChannelsDropout(probability=p, p_drop=float(params.get("p_drop", 0.2)), random_state=rs))
        elif name == "channels_shuffle":
            transforms.append(ChannelsShuffle(probability=p, p_shuffle=float(params.get("p_shuffle", 0.2)), random_state=rs))
        elif name == "channels_symmetry":
            # WARNING: swaps left/right hemisphere channels. For LEFT vs RIGHT hand MI this
            # effectively flips the class WITHOUT flipping the label -> mislabeled samples.
            # Use only as an experiment; expect it to HURT unless you also swap labels.
            transforms.append(ChannelsSymmetry(probability=p, ordered_ch_names=ch_names, random_state=rs))
        else:
            raise ValueError(f"Unknown augmentation transform '{name}'")

    print(f"[augmentation] '{aug_cfg.get('name', 'custom')}' -> {len(transforms)} transform(s), train iterator only")
    return transforms


### 3.6 Normalizer, classifier builder, metrics, collapse + probability diagnostics

In [12]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def dataset_to_array(dataset):
    X = np.stack([np.asarray(dataset[i][0], dtype=np.float32) for i in range(len(dataset))], axis=0)
    if X.ndim != 3:
        raise ValueError(f"Expected dataset windows to stack as N x C x T, got shape={X.shape}.")
    return X

def _safe_scale(scale, eps):
    scale = np.asarray(scale, dtype=np.float32)
    return np.where(np.abs(scale) < float(eps), 1.0, scale).astype(np.float32)

def _as_window_2d(x):
    """Keep each EEG example in C x T format for Braindecode SignalJEPA_PreLocal."""
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 3 and x.shape[0] == 1:
        x = x[0]
    if x.ndim != 2:
        raise ValueError(f"Expected one EEG window to have shape C x T, got shape={x.shape}.")
    return x

def _state_array_for_window(value, x):
    """Convert fitted fold statistics to shapes that broadcast over a single C x T window."""
    arr = np.asarray(value, dtype=np.float32)
    if arr.ndim == 0:
        return arr
    if x.ndim == 2 and arr.ndim == 3 and arr.shape[0] == 1:
        # Old shape from fold fitting was 1 x C x 1. For one example, use C x 1.
        arr = arr[0]
    if x.ndim == 2 and arr.ndim == 1 and arr.shape[0] == x.shape[0]:
        arr = arr[:, None]
    return arr

def apply_normalizer_to_array(x, state):
    """Apply a fitted fold normalizer or trial-wise normalizer to one C x T example.

    Important: this function must return C x T, not 1 x C x T.
    Returning 1 x C x T makes the DataLoader batch 4D and breaks SignalJEPA_PreLocal.
    """
    mode = str((state or {}).get("mode", "none")).lower()
    eps = float((state or {}).get("eps", CONFIG.get("normalization_eps", 1e-6)))
    x = _as_window_2d(x)

    if mode in ("none", "off", "false"):
        return x

    if mode == "train_global_zscore":
        mean = _state_array_for_window(state["mean"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - mean) / scale)

    if mode == "train_channel_zscore":
        mean = _state_array_for_window(state["mean"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - mean) / scale)

    if mode == "train_channel_robust":
        median = _state_array_for_window(state["median"], x)
        scale = _state_array_for_window(state["scale"], x)
        return _as_window_2d((x - median) / scale)

    if mode == "trial_global_zscore":
        mean = x.mean(keepdims=True)
        scale = max(float(x.std()), eps)
        return _as_window_2d((x - mean) / scale)

    if mode == "trial_channel_zscore":
        mean = x.mean(axis=-1, keepdims=True)
        scale = _safe_scale(x.std(axis=-1, keepdims=True), eps)
        return _as_window_2d((x - mean) / scale)

    raise ValueError(f"Unsupported normalization_mode={mode}")

def fit_fold_normalizer(train_dataset):
    """Fit normalization on the training fold only where applicable."""
    mode = str(CONFIG.get("normalization_mode", "none")).lower()
    eps = float(CONFIG.get("normalization_eps", 1e-6))
    state = {"mode": mode, "eps": eps}
    summary = {"mode": mode, "fit_scope": "none"}

    if mode in ("none", "off", "false", "trial_global_zscore", "trial_channel_zscore"):
        if mode.startswith("trial_"):
            summary["fit_scope"] = "trial_only_no_fold_fit"
        return state, summary

    X = dataset_to_array(train_dataset)  # N x C x T

    if mode == "train_global_zscore":
        mean = np.asarray(X.mean(), dtype=np.float32)
        scale = np.asarray(max(float(X.std()), eps), dtype=np.float32)
        state.update({"mean": mean, "scale": scale})
        summary.update({
            "fit_scope": "training_fold",
            "mean_shape": [],
            "scale_shape": [],
            "train_mean": float(mean),
            "train_scale": float(scale),
        })
        return state, summary

    if mode == "train_channel_zscore":
        # Shape is C x 1 so it broadcasts correctly over one C x T window.
        mean = X.mean(axis=(0, 2)).astype(np.float32)[:, None]
        scale = _safe_scale(X.std(axis=(0, 2)).astype(np.float32)[:, None], eps)
        state.update({"mean": mean, "scale": scale})
        summary.update({
            "fit_scope": "training_fold",
            "mean_shape": list(mean.shape),
            "scale_shape": list(scale.shape),
            "mean_scale_min": float(scale.min()),
            "mean_scale_max": float(scale.max()),
        })
        return state, summary

    if mode == "train_channel_robust":
        # Shape is C x 1 so it broadcasts correctly over one C x T window.
        median = np.median(X, axis=(0, 2)).astype(np.float32)[:, None]
        q75 = np.percentile(X, 75, axis=(0, 2)).astype(np.float32)[:, None]
        q25 = np.percentile(X, 25, axis=(0, 2)).astype(np.float32)[:, None]
        iqr = _safe_scale((q75 - q25).astype(np.float32), eps)
        state.update({"median": median, "scale": iqr})
        summary.update({
            "fit_scope": "training_fold",
            "median_shape": list(median.shape),
            "scale_shape": list(iqr.shape),
            "iqr_min": float(iqr.min()),
            "iqr_max": float(iqr.max()),
        })
        return state, summary

    raise ValueError(f"Unsupported normalization_mode={CONFIG.get('normalization_mode')}")

def maybe_wrap_normalized(train_set, test_set):
    state, summary = fit_fold_normalizer(train_set)
    mode = str(summary.get("mode", "none")).lower()
    if mode in ("none", "off", "false"):
        return train_set, test_set, summary
    return FoldNormalizedDataset(train_set, state), FoldNormalizedDataset(test_set, state), summary


class BalancedLabelSmoothingLoss(torch.nn.Module):
    def __init__(self, label_smoothing=0.0, prediction_balance_loss_weight=0.0):
        super().__init__()
        self.label_smoothing = float(label_smoothing)
        self.prediction_balance_loss_weight = float(prediction_balance_loss_weight)

    def forward(self, y_pred, y_true):
        if isinstance(y_pred, (tuple, list)):
            y_pred = y_pred[0]

        log_probs = torch.nn.functional.log_softmax(y_pred, dim=1)
        n_classes = int(log_probs.shape[1])

        y_true = y_true.long()
        nll = -log_probs.gather(1, y_true.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_probs.mean(dim=1)

        smoothing = self.label_smoothing
        loss = ((1.0 - smoothing) * nll + smoothing * smooth_loss).mean()

        if self.prediction_balance_loss_weight > 0:
            probs = torch.softmax(y_pred, dim=1)
            mean_probs = probs.mean(dim=0)
            target_probs = torch.full_like(mean_probs, 1.0 / n_classes)
            balance_loss = ((mean_probs - target_probs) ** 2).sum()
            loss = loss + self.prediction_balance_loss_weight * balance_loss

        return loss

def make_train_split():
    val_split = CONFIG["val_split"]
    if val_split is None or float(val_split) <= 0.0:
        return None
    return ValidSplit(
        cv=float(val_split),
        stratified=True,
        random_state=int(CONFIG.get("val_split_random_state", CONFIG.get("seed", 2026))),
    )

def make_callbacks(max_epochs=None):
    callbacks = []
    train_split = make_train_split()
    patience = CONFIG["early_stopping_patience"]
    checkpoint_metric = str(CONFIG.get("checkpoint_metric", "valid_loss")).lower()

    if train_split is not None and checkpoint_metric == "valid_balanced_accuracy":
        callbacks.append(
            (
                "valid_balanced_accuracy",
                EpochScoring(
                    scoring="balanced_accuracy",
                    lower_is_better=False,
                    on_train=False,
                    name="valid_balanced_accuracy",
                ),
            )
        )

    if train_split is not None and patience is not None and int(patience) > 0:
        if checkpoint_metric == "valid_balanced_accuracy":
            callbacks.append(
                (
                    "early_stopping",
                    EarlyStopping(
                        monitor="valid_balanced_accuracy",
                        patience=int(patience),
                        lower_is_better=False,
                        load_best=True,
                    ),
                )
            )
        else:
            callbacks.append(
                (
                    "early_stopping",
                    EarlyStopping(
                        monitor="valid_loss",
                        patience=int(patience),
                        lower_is_better=True,
                        load_best=True,
                    ),
                )
            )

    clip_norm = CONFIG.get("gradient_clip_norm", None)
    if clip_norm is not None and float(clip_norm) > 0:
        try:
            from skorch.callbacks import GradientNormClipping
        except Exception as exc:
            raise RuntimeError("gradient_clip_norm was set but skorch.callbacks.GradientNormClipping could not be imported.") from exc
        callbacks.append(("grad_clip", GradientNormClipping(float(clip_norm))))

    return callbacks

def build_classifier(model, callbacks, max_epochs, fold_seed=None, warm_start=False):
    train_generator = None
    if fold_seed is not None:
        train_generator = torch.Generator()
        train_generator.manual_seed(fold_seed)

    optimizer_name = str(CONFIG.get("optimizer_name", "adam")).lower()
    if optimizer_name == "adam":
        optimizer_cls = torch.optim.Adam
    elif optimizer_name == "adamw":
        optimizer_cls = torch.optim.AdamW
    else:
        raise ValueError(f"Unsupported optimizer_name={CONFIG.get('optimizer_name')}")

    clf_kwargs = {
        "batch_size": int(CONFIG["batch_size"]),
        "max_epochs": int(max_epochs),
        "device": DEVICE,
        "callbacks": callbacks,
        "train_split": make_train_split(),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
        "optimizer": optimizer_cls,
        "warm_start": warm_start,
        "criterion": BalancedLabelSmoothingLoss,
        "criterion__label_smoothing": float(CONFIG.get("label_smoothing", 0.0)),
        "criterion__prediction_balance_loss_weight": float(CONFIG.get("prediction_balance_loss_weight", 0.0)),
    }
    if CONFIG["learning_rate"] is not None:
        clf_kwargs["lr"] = float(CONFIG["learning_rate"])
    if CONFIG.get("weight_decay", None) is not None:
        clf_kwargs["optimizer__weight_decay"] = float(CONFIG.get("weight_decay", 0.0))
    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator
    # --- braindecode on-the-fly augmentation (TRAIN iterator only; validation stays clean) ---
    _aug_transforms = build_braindecode_transforms(CONFIG.get("augmentation", {}))
    if _aug_transforms:
        from braindecode.augmentation import AugmentedDataLoader
        clf_kwargs["iterator_train"] = AugmentedDataLoader
        clf_kwargs["iterator_train__transforms"] = _aug_transforms

    return EEGClassifier(model, **clf_kwargs)

def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

def compute_prediction_probability_diagnostics(clf, test_set, y_pred, n_classes):
    if not bool(CONFIG.get("log_probability_diagnostics", True)):
        return None
    try:
        probs = np.asarray(clf.predict_proba(test_set), dtype=float)
    except Exception as exc:
        return {"available": False, "reason": f"predict_proba failed: {exc}"}

    if probs.ndim != 2 or probs.shape[0] != len(test_set):
        return {"available": False, "reason": f"Unexpected probability shape: {list(probs.shape)}"}
    if not np.isfinite(probs).all():
        return {"available": False, "reason": "Non-finite probabilities."}

    row_sums = probs.sum(axis=1, keepdims=True)
    if np.any(probs < 0) or not np.allclose(row_sums, 1.0, atol=1e-3):
        exp_probs = np.exp(probs - probs.max(axis=1, keepdims=True))
        probs = exp_probs / np.maximum(exp_probs.sum(axis=1, keepdims=True), 1e-12)

    eps = 1e-12
    confidence = probs.max(axis=1)
    entropy = -np.sum(probs * np.log(probs + eps), axis=1)
    normalized_entropy = entropy / np.log(max(probs.shape[1], 2))
    predicted_class_probability = probs[np.arange(len(probs)), np.asarray(y_pred, dtype=int)]

    return {
        "available": True,
        "probability_shape": list(probs.shape),
        "mean_probability_by_class": _json_safe_float_list(probs.mean(axis=0)),
        "std_probability_by_class": _json_safe_float_list(probs.std(axis=0)),
        "mean_confidence": _json_safe_float(confidence.mean()),
        "std_confidence": _json_safe_float(confidence.std()),
        "mean_prediction_entropy": _json_safe_float(entropy.mean()),
        "mean_normalized_prediction_entropy": _json_safe_float(normalized_entropy.mean()),
        "mean_predicted_class_probability": _json_safe_float(predicted_class_probability.mean()),
    }

def compute_collapse_diagnostics(y_pred, n_classes):
    pred_hist = np.bincount(np.asarray(y_pred, dtype=int), minlength=n_classes)
    n_pred = int(pred_hist.sum())
    collapse_ratio = float(pred_hist.max() / n_pred) if n_pred else 0.0
    threshold = float(CONFIG.get("collapse_threshold", 0.90))
    return {
        "prediction_histogram": pred_hist.tolist(),
        "collapse_ratio": _json_safe_float(collapse_ratio),
        "collapse_threshold": threshold,
        "collapse_flag": bool(collapse_ratio >= threshold),
        "majority_predicted_class": int(pred_hist.argmax()) if n_pred else None,
    }

### 3.7 Evaluation-split builder (same protocol/seeds as reference)

In [13]:
def make_evaluation_splits(y, n_classes):
    """Create subject-level evaluation splits from CONFIG.

    Supported modes:
      - stratified_kfold: S-JEPA-style 5-fold within-subject CV
      - liu2024_repeated_60_40: 10 repeated stratified 60/40 train/test splits
      - repeated_stratified_split: generic repeated stratified holdout
    """
    y = np.asarray(y, dtype=np.int64)
    counts = np.bincount(y, minlength=n_classes)
    indices = np.arange(len(y))
    mode = str(CONFIG.get("evaluation_mode", "stratified_kfold")).lower()

    if mode == "stratified_kfold":
        n_folds = int(CONFIG.get("cv_folds", 5))
        if counts.min() < n_folds:
            raise ValueError(f"Cannot use {n_folds} folds with class counts={counts.tolist()}.")

        splitter = StratifiedKFold(
            n_splits=n_folds,
            shuffle=True,
            random_state=int(CONFIG.get("cv_random_state", CONFIG.get("seed", 2026))),
        )
        splits = []
        for fold_id, (train_idx, test_idx) in enumerate(splitter.split(indices, y), start=1):
            if CONFIG.get("assert_balanced_folds", True) and np.all(counts % n_folds == 0):
                expected_test = (counts // n_folds).astype(int)
                expected_train = (counts - expected_test).astype(int)
                train_counts = np.bincount(y[train_idx], minlength=n_classes)
                test_counts = np.bincount(y[test_idx], minlength=n_classes)
                assert np.array_equal(train_counts, expected_train), (
                    f"Unexpected train class counts for fold {fold_id}: {train_counts.tolist()} != {expected_train.tolist()}"
                )
                assert np.array_equal(test_counts, expected_test), (
                    f"Unexpected test class counts for fold {fold_id}: {test_counts.tolist()} != {expected_test.tolist()}"
                )
            splits.append({
                "split_id": int(fold_id),
                "fold_id": int(fold_id),
                "idx_train": train_idx,
                "idx_test": test_idx,
                "evaluation_mode": mode,
                "evaluation_protocol": f"{n_folds}-fold stratified within-subject CV",
                "n_total_splits": int(n_folds),
            })
        return splits

    if mode in ("liu2024_repeated_60_40", "repeated_stratified_split"):
        n_repeats = int(CONFIG.get("n_repeats", 10))
        test_size = float(CONFIG.get("test_size", 0.40))
        splitter = StratifiedShuffleSplit(
            n_splits=n_repeats,
            test_size=test_size,
            random_state=int(CONFIG.get("split_random_state", CONFIG.get("seed", 2026))),
        )
        splits = []
        for split_id, (train_idx, test_idx) in enumerate(splitter.split(indices, y), start=1):
            train_counts = np.bincount(y[train_idx], minlength=n_classes)
            test_counts = np.bincount(y[test_idx], minlength=n_classes)
            if CONFIG.get("assert_balanced_folds", True):
                if train_counts.min() < 1 or test_counts.min() < 1:
                    raise ValueError(
                        f"Invalid stratified holdout split {split_id}: train_counts={train_counts.tolist()}, "
                        f"test_counts={test_counts.tolist()}"
                    )
            protocol = "Liu2024-style 10 repeated stratified 60/40 train/test splits" if mode == "liu2024_repeated_60_40" else f"{n_repeats} repeated stratified holdout splits, test_size={test_size}"
            splits.append({
                "split_id": int(split_id),
                "fold_id": int(split_id),
                "idx_train": train_idx,
                "idx_test": test_idx,
                "evaluation_mode": mode,
                "evaluation_protocol": protocol,
                "n_total_splits": int(n_repeats),
                "test_size": float(test_size),
            })
        return splits

    raise ValueError(f"Unsupported evaluation_mode={CONFIG.get('evaluation_mode')}")

# Backward compatibility.
def make_fold_splits(y, n_folds, n_classes):
    old_cv_folds = CONFIG.get("cv_folds")
    old_mode = CONFIG.get("evaluation_mode")
    CONFIG["evaluation_mode"] = "stratified_kfold"
    CONFIG["cv_folds"] = int(n_folds)
    try:
        return make_evaluation_splits(y, n_classes)
    finally:
        CONFIG["cv_folds"] = old_cv_folds
        CONFIG["evaluation_mode"] = old_mode


### 3.8 Results aggregation helper

In [14]:
def aggregate_results(fold_results):
    grouped = {}
    for result in fold_results:
        sid = result.get("subject_id", "global")
        grouped.setdefault(sid, {"accuracies": [], "balanced_accuracies": []})
        grouped[sid]["accuracies"].append(result.get("accuracy"))
        grouped[sid]["balanced_accuracies"].append(result.get("balanced_accuracy"))

    for sid, metrics in grouped.items():
        acc_values = [v for v in metrics["accuracies"] if v is not None]
        bal_values = [v for v in metrics["balanced_accuracies"] if v is not None]
        metrics["mean_accuracy"] = float(np.mean(acc_values)) if acc_values else None
        metrics["std_accuracy"] = float(np.std(acc_values)) if acc_values else None
        metrics["mean_balanced_accuracy"] = float(np.mean(bal_values)) if bal_values else None
        metrics["std_balanced_accuracy"] = float(np.std(bal_values)) if bal_values else None

    all_accs = [r["accuracy"] for r in fold_results if r.get("accuracy") is not None]
    all_bals = [r["balanced_accuracy"] for r in fold_results if r.get("balanced_accuracy") is not None]
    global_metrics = {
        "mean_accuracy": float(np.mean(all_accs)) if all_accs else None,
        "std_accuracy": float(np.std(all_accs)) if all_accs else None,
        "mean_balanced_accuracy": float(np.mean(all_bals)) if all_bals else None,
        "std_balanced_accuracy": float(np.std(all_bals)) if all_bals else None,
        "n_subjects": len(grouped),
        "n_folds_total": len(fold_results),
    }
    return grouped, global_metrics


# 4. Load data (carried verbatim)

Builds `X_ALL`, `Y_ALL`, `SUBJECT_ID_ALL`, and `SUBJECT_WINDOWS` with the reference
preprocessing. Requires the Liu2024 `.mat` files and MNE.

In [15]:
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
MAT_FILES = []

files = find_source_mat_files(SOURCE_EXTRACT_DIR)

if files:
    MAT_FILES = files

if not MAT_FILES:
    raise FileNotFoundError(
        "Could not find Liu2024 source .mat files. "
    )

print(f"Source extract dir: {SOURCE_EXTRACT_DIR}")
print(f"Found {len(MAT_FILES)} .mat files")

# Write a structure preview for the first subject. This makes it clear whether
# the local files expose top-level rawdata/labels or an `eeg` struct.
preview_path = ARTIFACT_DIR / "mat_structure_preview_first_subject.csv"
mat_structure_preview(MAT_FILES[0]).to_csv(preview_path, index=False)
print(f"MAT structure preview saved to: {preview_path}")

subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if CONFIG["subjects_to_use"] is not None and sid not in set(int(s) for s in CONFIG["subjects_to_use"]):
        continue
    if sid in set(int(s) for s in CONFIG["exclude_subjects"]):
        continue
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    validate_liu_source_subject(X_raw, y_raw, sid, path=p)
    subjects.append({
        "subject_id": sid,
        "path": str(p),
        "rawdata_shape": tuple(X_raw.shape),
        "labels_shape": tuple(y_raw.shape),
        "label_counts_raw": np.bincount(y_raw.astype(int), minlength=3).tolist(),
        "raw_field": raw_field,
        "label_field": label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values("subject_id").reset_index(drop=True)
if subjects_df.empty:
    raise RuntimeError("No subjects loaded.")

SUBJECTS = [int(s) for s in subjects_df["subject_id"].tolist()]
print(f"Subjects loaded: {SUBJECTS}")

subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
subjects_df.to_csv(subject_inventory_path, index=False)
print(f"Subject inventory saved to: {subject_inventory_path}")

EEG_INFO = make_liu_info(EFFECTIVE_SFREQ)
CHS_INFO = EEG_INFO["chs"]
CH_NAMES = list(EEG_CHANNEL_NAMES)

Xs, ys, subject_ids = [], [], []
window_summary_rows = []
preprocessing_steps_first_subject = None

for item in subjects_df.to_dict("records"):
    sid = int(item["subject_id"])
    X_raw, y_raw, _, _ = load_subject_mat(Path(item["path"]))
    X_win, y, samples_per_trial, preprocessing_steps, preprocessing_stats = preprocess_subject_configurable(X_raw, y_raw, sid)

    if preprocessing_steps_first_subject is None:
        preprocessing_steps_first_subject = preprocessing_steps

    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))
    window_summary_rows.append({
        "subject_id": sid,
        "n_windows": int(len(y)),
        "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
        "preprocessed_shape": tuple(X_win.shape),
        "resampled_samples_per_trial": int(samples_per_trial),
        "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
        "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
        "target_window_samples": int(WINDOW_SAMPLES),
        "effective_window_duration_s": float(TARGET_TRIAL_DURATION_S),
        "source_unit": CONFIG["source_unit"],
        "final_model_unit": CONFIG["final_model_unit"],
        "demean_mode": CONFIG["demean_mode"],
        "reference_mode": CONFIG["reference_mode"],
        "reference_timing": CONFIG["reference_timing"],
        "resample": bool(CONFIG["resample"]),
        "effective_sfreq": float(EFFECTIVE_SFREQ),
        "filter_enabled": bool(CONFIG["filter_enabled"]),
        "filter_low": CONFIG.get("filter_low"),
        "filter_high": CONFIG.get("filter_high"),
        "filter_method": CONFIG.get("filter_method"),
        "mi_window_start_s": float(CONFIG["mi_window_start_s"]),
        "eog_correction": CONFIG.get("eog_correction"),
        "artifact_clip_mode": CONFIG.get("artifact_clip_mode"),
        "reject_bad_trials": bool(CONFIG.get("reject_bad_trials", False)),
        "normalization_mode": CONFIG.get("normalization_mode"),
        **preprocessing_stats,
    })

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0)
SUBJECT_ID_ALL = np.asarray(subject_ids)
print(f"X_ALL shape: {X_ALL.shape} | Y_ALL counts: {np.bincount(Y_ALL).tolist()}")

if preprocessing_steps_first_subject is not None:
    print("Preprocessing steps used:")
    for step in preprocessing_steps_first_subject:
        print(f"  - {step}")

window_summary_df = pd.DataFrame(window_summary_rows).sort_values("subject_id")
window_summary_path = ARTIFACT_DIR / "window_counts_by_subject.csv"
window_summary_df.to_csv(window_summary_path, index=False)
print(f"Window summary saved to: {window_summary_path}")
display(window_summary_df.head())

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

SUBJECT_WINDOWS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    idx = np.where(SUBJECT_ID_ALL == sid)[0]
    SUBJECT_WINDOWS[str(sid)] = SubjectArrayDataset(X_ALL[idx], Y_ALL[idx], subject_id=sid)


[2026-06-13 09:48:41] Source extract dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_data/liu2024_figshare/sourcedata
[2026-06-13 09:48:41] Found 50 .mat files
[2026-06-13 09:48:41] MAT structure preview saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-supervised-classifier-diagnostics/20260613_0948_d81691de/mat_structure_preview_first_subject.csv
[2026-06-13 09:48:47] Subjects loaded: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
[2026-06-13 09:48:47] Subject inventory saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-supervised-classifier-diagnostics/20260613_0948_d81691de/subject_inventory.csv
[2026-06-13 09:49:01] X_ALL shape: (2000, 29, 537) | Y_ALL counts: [1000, 1000]
[2026

,subject_id,n_windows,class_counts,preprocessed_shape,resampled_samples_per_trial,crop_start_sample,crop_stop_sample,target_window_samples,effective_window_duration_s,source_unit,...,reject_bad_trials,normalization_mode,artifact_clip_applied,artifact_clip_threshold,n_trials_before_reject,n_trials_after_reject,n_rejected_trials,rejection_skipped,pipeline_plan,runtime_steps
0,1,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
1,2,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
2,3,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
3,4,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
4,5,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...


# 5. Label & split sanity checks (Q1, Q2)

Before any modeling: confirm labels are the expected two classes with the documented
{1,2}→{0,1} mapping, summarize per-subject class counts, and verify the within-subject
stratified folds are balanced.

In [16]:
# Per-subject arrays for the classical baselines (kept in memory; identical to SUBJECT_WINDOWS).
SUBJECT_ARRAYS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    m = (SUBJECT_ID_ALL == sid)
    SUBJECT_ARRAYS[str(sid)] = (X_ALL[m].astype(np.float64), Y_ALL[m].astype(int))

print("Label inventory")
print(f"  unique labels in Y_ALL (zero-based): {sorted(np.unique(Y_ALL).tolist())}  "
      f"(source {{1,2}} mapped to {{0,1}} by labels_to_zero_based)")
print(f"  global class counts: {np.bincount(Y_ALL, minlength=TARGET_N_CLASSES).tolist()}")
print(f"  class_names (VERIFY mapping vs dataset docs): {CONFIG['class_names']}")

rows = []
balance_ok = True
for sid, (X, y) in SUBJECT_ARRAYS.items():
    counts = np.bincount(y, minlength=TARGET_N_CLASSES)
    splits = make_evaluation_splits(y, TARGET_N_CLASSES)
    fold_bal = []
    for s in splits:
        tr = np.bincount(y[s["idx_train"]], minlength=TARGET_N_CLASSES)
        te = np.bincount(y[s["idx_test"]], minlength=TARGET_N_CLASSES)
        fold_bal.append((tr.tolist(), te.tolist()))
        if te.min() == 0 or tr.min() == 0:
            balance_ok = False
    rows.append({"subject_id": sid, "n_trials": int(len(y)),
                 "class_counts": counts.tolist(),
                 "n_folds": len(splits),
                 "min_test_per_class": int(min(min(te) for _, te in fold_bal))})
label_split_df = pd.DataFrame(rows).sort_values("subject_id")
try:
    from IPython.display import display
    display(label_split_df)
except Exception:
    print(label_split_df.to_string(index=False))
print(f"\nAll folds have >=1 sample per class in train & test: {balance_ok}")
label_split_df.to_csv(ARTIFACT_DIR / "label_split_sanity.csv", index=False)


[2026-06-13 09:49:02] Label inventory
[2026-06-13 09:49:02]   unique labels in Y_ALL (zero-based): [0, 1]  (source {1,2} mapped to {0,1} by labels_to_zero_based)
[2026-06-13 09:49:02]   global class counts: [1000, 1000]
[2026-06-13 09:49:02]   class_names (VERIFY mapping vs dataset docs): ['class_0', 'class_1']


,subject_id,n_trials,class_counts,n_folds,min_test_per_class
0,1,40,"[20, 20]",5,4
9,10,40,"[20, 20]",5,4
10,11,40,"[20, 20]",5,4
11,12,40,"[20, 20]",5,4
12,13,40,"[20, 20]",5,4
13,14,40,"[20, 20]",5,4
14,15,40,"[20, 20]",5,4
15,16,40,"[20, 20]",5,4
16,17,40,"[20, 20]",5,4
17,18,40,"[20, 20]",5,4



[2026-06-13 09:49:02] All folds have >=1 sample per class in train & test: True


# 6. Features and the per-method fold runner

Feature extractors and a single dispatch that fits each baseline on the training split and
predicts on the test split, returning a result dict in the **same schema** the reference uses
(so aggregation and diagnostics are shared). Everything fold-safe.

In [17]:
def _bandpass(x, lo, hi, sfreq, order=4):
    """Zero-phase Butterworth bandpass on (..., n_times). Falls back to raw on failure."""
    nyq = 0.5 * sfreq
    lo_n, hi_n = max(lo / nyq, 1e-4), min(hi / nyq, 0.999)
    if hi_n <= lo_n:
        return x
    b, a = signal.butter(order, [lo_n, hi_n], btype="band")
    return signal.filtfilt(b, a, x, axis=-1)

def extract_features(X, spec):
    """X: (n_trials, n_chans, n_times) -> (n_trials, n_features). Interpretable log-power."""
    feat = str(spec.get("feature", "logvar"))
    eps = 1e-8
    if feat == "logvar":
        return np.log(np.var(X, axis=-1) + eps)                      # (n_trials, n_chans)
    if feat == "multiband_logvar":
        blocks = []
        for lo, hi in spec.get("bands", [[8, 12], [12, 30]]):
            Xb = _bandpass(X, lo, hi, EFFECTIVE_SFREQ)
            blocks.append(np.log(np.var(Xb, axis=-1) + eps))
        return np.concatenate(blocks, axis=1)
    raise ValueError(f"Unknown feature: {feat}")

def prob_diagnostics_from_probs(probs, y_pred, n_classes):
    """Array version of the reference's probability diagnostics (works for any estimator)."""
    if probs is None:
        return {"available": False, "reason": "estimator did not provide probabilities"}
    probs = np.asarray(probs, dtype=float)
    if probs.ndim != 2 or not np.isfinite(probs).all():
        return {"available": False, "reason": f"bad probability array shape={list(probs.shape)}"}
    row = probs.sum(axis=1, keepdims=True)
    if np.any(probs < 0) or not np.allclose(row, 1.0, atol=1e-3):
        e = np.exp(probs - probs.max(axis=1, keepdims=True))
        probs = e / np.maximum(e.sum(axis=1, keepdims=True), 1e-12)
    eps = 1e-12
    conf = probs.max(axis=1)
    ent = -np.sum(probs * np.log(probs + eps), axis=1)
    norm_ent = ent / np.log(max(probs.shape[1], 2))
    pcp = probs[np.arange(len(probs)), np.asarray(y_pred, dtype=int)]
    return {"available": True, "probability_shape": list(probs.shape),
            "mean_probability_by_class": _json_safe_float_list(probs.mean(axis=0)),
            "std_probability_by_class": _json_safe_float_list(probs.std(axis=0)),
            "mean_confidence": _json_safe_float(conf.mean()),
            "std_confidence": _json_safe_float(conf.std()),
            "frac_confidence_over_0p9": _json_safe_float(float(np.mean(conf > 0.9))),
            "mean_prediction_entropy": _json_safe_float(ent.mean()),
            "mean_normalized_prediction_entropy": _json_safe_float(norm_ent.mean()),
            "mean_predicted_class_probability": _json_safe_float(pcp.mean()),
            "_probs": probs.tolist()}  # retained for the calibration curve; dropped before JSON save

def _build_eegnet():
    n_chans, n_times, n_out = len(CH_NAMES), int(WINDOW_SAMPLES), int(TARGET_N_CLASSES)
    for kwargs in ({"n_chans": n_chans, "n_outputs": n_out, "n_times": n_times},
                   {"n_chans": n_chans, "n_outputs": n_out, "n_times": n_times,
                    "chs_info": CHS_INFO, "sfreq": EFFECTIVE_SFREQ}):
        try:
            return EEGNetv4(**kwargs)
        except Exception:
            continue
    return EEGNetv4(n_chans, n_out, n_times)  # last resort positional

def run_fold_for_method(method_name, spec, Xtr, ytr, Xte, yte):
    """Fit on train, predict on test. Returns (y_pred, probs_or_None, info)."""
    t = spec["type"]
    if t == "sklearn_features":
        Ftr, Fte = extract_features(Xtr, spec), extract_features(Xte, spec)
        scaler = StandardScaler().fit(Ftr)                 # fold-safe: fit on train only
        clf = LogisticRegression(max_iter=2000, C=1.0)
        clf.fit(scaler.transform(Ftr), ytr)
        Z = scaler.transform(Fte)
        return clf.predict(Z), clf.predict_proba(Z), {"n_features": Ftr.shape[1]}
    if t == "csp":
        if not HAVE_CSP:
            raise RuntimeError("mne CSP unavailable")
        csp = CSP(n_components=int(spec.get("n_components", 4)), reg=None, log=True, norm_trace=False)
        csp.fit(Xtr, ytr)                                  # fold-safe
        clf = LinearDiscriminantAnalysis().fit(csp.transform(Xtr), ytr)
        Tte = csp.transform(Xte)
        return clf.predict(Tte), clf.predict_proba(Tte), {"n_components": int(spec.get("n_components", 4))}
    if t == "fbcsp":
        if not HAVE_CSP:
            raise RuntimeError("mne CSP unavailable")
        nc = int(spec.get("n_components", 4))
        feats_tr, feats_te, csps = [], [], 0
        for lo, hi in spec.get("bands", [[8, 12], [12, 30]]):
            Xtr_b, Xte_b = _bandpass(Xtr, lo, hi, EFFECTIVE_SFREQ), _bandpass(Xte, lo, hi, EFFECTIVE_SFREQ)
            try:
                csp = CSP(n_components=nc, reg="ledoit_wolf", log=True, norm_trace=False)
                csp.fit(Xtr_b, ytr)
                feats_tr.append(csp.transform(Xtr_b)); feats_te.append(csp.transform(Xte_b)); csps += 1
            except Exception:
                continue
        if not feats_tr:
            raise RuntimeError("FBCSP produced no usable bands")
        Ftr, Fte = np.concatenate(feats_tr, axis=1), np.concatenate(feats_te, axis=1)
        scaler = StandardScaler().fit(Ftr)
        clf = LinearDiscriminantAnalysis().fit(scaler.transform(Ftr), ytr)
        Z = scaler.transform(Fte)
        return clf.predict(Z), clf.predict_proba(Z), {"n_bands_used": csps, "n_features": Ftr.shape[1]}
    if t == "riemann":
        if not HAVE_PYRIEMANN:
            raise RuntimeError("pyriemann unavailable")
        cov = Covariances(estimator=str(spec.get("cov_estimator", "oas")))
        Ctr, Cte = cov.fit_transform(Xtr), cov.transform(Xte)
        clf = MDM().fit(Ctr, ytr)
        try:
            probs = clf.predict_proba(Cte)
        except Exception:
            probs = None
        return clf.predict(Cte), probs, {"cov_estimator": str(spec.get("cov_estimator", "oas"))}
    if t == "eegnet":
        if not (HAVE_BRAINDECODE and HAVE_TORCH):
            raise RuntimeError("braindecode/torch unavailable")
        if CONFIG["set_seed"]:
            seed_everything(BASE_SEED)
        model = _build_eegnet()
        clf = build_classifier(model, callbacks=make_callbacks(int(CONFIG["n_epochs"])),
                               max_epochs=int(CONFIG["n_epochs"]), fold_seed=BASE_SEED, warm_start=False)
        train_ds = SubjectArrayDataset(Xtr.astype(np.float32), ytr)
        test_ds = SubjectArrayDataset(Xte.astype(np.float32), yte)
        clf.fit(train_ds, y=ytr)
        y_pred = clf.predict(test_ds)
        try:
            probs = np.asarray(clf.predict_proba(test_ds), dtype=float)
        except Exception:
            probs = None
        return y_pred, probs, {"n_epochs": int(CONFIG["n_epochs"])}
    raise ValueError(f"Unknown method type: {t}")

def evaluate_method_on_subject(method_name, spec, sid, X, y):
    splits = make_evaluation_splits(y, TARGET_N_CLASSES)   # identical protocol/seed to reference
    out = []
    for s in splits:
        Xtr, ytr = X[s["idx_train"]], y[s["idx_train"]]
        Xte, yte = X[s["idx_test"]], y[s["idx_test"]]
        y_pred, probs, info = run_fold_for_method(method_name, spec, Xtr, ytr, Xte, yte)
        y_pred = np.asarray(y_pred, dtype=int)
        metrics = compute_classification_metrics(yte, y_pred)
        collapse = compute_collapse_diagnostics(y_pred, TARGET_N_CLASSES)
        probd = prob_diagnostics_from_probs(probs, y_pred, TARGET_N_CLASSES)
        out.append({
            "method": method_name, "subject_id": str(sid), "fold_id": int(s["split_id"]),
            "n_train": int(len(ytr)), "n_test": int(len(yte)),
            "train_class_counts": np.bincount(ytr, minlength=TARGET_N_CLASSES).tolist(),
            "test_class_counts": np.bincount(yte, minlength=TARGET_N_CLASSES).tolist(),
            "accuracy": metrics["accuracy"], "balanced_accuracy": metrics["balanced_accuracy"],
            "confusion_matrix": confusion_matrix(yte, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist(),
            "prediction_histogram": collapse["prediction_histogram"],
            "collapse_diagnostics": collapse, "probability_diagnostics": probd,
            "evaluation_mode": s["evaluation_mode"], "evaluation_protocol": s["evaluation_protocol"],
            "fit_info": info,
        })
    return out


# 7. Run the baselines

Each enabled method (whose dependencies are present) is evaluated with the identical
within-subject CV across all subjects. Per-method artifacts are written to a subfolder.

In [18]:
def method_available(spec):
    t = spec["type"]
    if t in ("csp", "fbcsp"): return HAVE_CSP
    if t == "riemann": return HAVE_PYRIEMANN
    if t == "eegnet": return HAVE_BRAINDECODE and HAVE_TORCH
    return True  # sklearn_features

def strip_probs(fold_results):
    """Remove the bulky per-sample probability arrays before JSON serialization."""
    clean = copy.deepcopy(fold_results)
    for r in clean:
        if isinstance(r.get("probability_diagnostics"), dict):
            r["probability_diagnostics"].pop("_probs", None)
    return clean

def method_diagnostics(fold_results, n_classes, pos_idx):
    bal = [r["balanced_accuracy"] for r in fold_results if r.get("balanced_accuracy") is not None]
    flags = [bool(r["collapse_diagnostics"]["collapse_flag"]) for r in fold_results]
    ratios = [r["collapse_diagnostics"]["collapse_ratio"] for r in fold_results]
    conf = [r["probability_diagnostics"].get("mean_confidence") for r in fold_results
            if r.get("probability_diagnostics", {}).get("available")]
    sat = [r["probability_diagnostics"].get("frac_confidence_over_0p9") for r in fold_results
           if r.get("probability_diagnostics", {}).get("available")]
    tot = np.zeros(n_classes)
    for r in fold_results:
        tot += np.asarray(r["prediction_histogram"], dtype=float)
    by_sub = {}
    for r in fold_results:
        by_sub.setdefault(r["subject_id"], []).append(r["balanced_accuracy"])
    sub_means = [float(np.mean(v)) for v in by_sub.values() if v]
    return {
        "n_folds": len(fold_results), "n_subjects": len(sub_means),
        "mean_balanced_accuracy": float(np.mean(bal)) if bal else None,
        "std_balanced_accuracy": float(np.std(bal)) if bal else None,
        "fold_balacc_std": float(np.std(bal)) if bal else None,
        "collapse_rate": float(np.mean(flags)) if flags else None,
        "mean_collapse_ratio": float(np.mean(ratios)) if ratios else None,
        "positive_class_pred_rate": float(tot[pos_idx] / tot.sum()) if tot.sum() else None,
        "subject_balacc_std": float(np.std(sub_means)) if sub_means else None,
        "subject_balacc_min": float(np.min(sub_means)) if sub_means else None,
        "mean_confidence": float(np.mean(conf)) if conf else None,
        "frac_confidence_over_0p9": float(np.mean(sat)) if sat else None,
    }

ALL_RESULTS = {}          # method -> list of fold dicts (with _probs retained in-memory)
METHOD_SUMMARY = {}
pos_idx = int(CONFIG["positive_class_index"])

for method_name, spec in CONFIG["methods"].items():
    if not spec.get("enabled", False):
        continue
    if not method_available(spec):
        print(f"[skip] {method_name}: required dependency not installed.")
        continue
    print("=" * 70); print(f"METHOD: {method_name} ({spec['type']})"); print("=" * 70)
    fold_results = []
    for sid in sorted(SUBJECT_ARRAYS.keys(), key=_sort_subject_key):
        X, y = SUBJECT_ARRAYS[sid]
        try:
            res = evaluate_method_on_subject(method_name, spec, sid, X, y)
            fold_results.extend(res)
            bals = [r["balanced_accuracy"] for r in res]
            print(f"  subject {sid}: bal_acc={np.mean(bals):.3f}±{np.std(bals):.3f}")
        except Exception as exc:
            print(f"  subject {sid}: FAILED ({exc})")
    if not fold_results:
        print(f"[skip] {method_name}: no results."); continue
    ALL_RESULTS[method_name] = fold_results
    subj_metrics, global_metrics = aggregate_results(fold_results)
    diag = method_diagnostics(fold_results, TARGET_N_CLASSES, pos_idx)
    METHOD_SUMMARY[method_name] = {"global_metrics": global_metrics, "diagnostics": diag}

    mdir = ARTIFACT_DIR / f"method_{method_name}"; mdir.mkdir(parents=True, exist_ok=True)
    with open(mdir / "cv_results.json", "w") as f:
        json.dump(strip_probs(fold_results), f, indent=2, default=str)
    with open(mdir / "subject_metrics.json", "w") as f:
        json.dump(subj_metrics, f, indent=2, default=str)
    with open(mdir / "global_metrics.json", "w") as f:
        json.dump(global_metrics, f, indent=2, default=str)
    with open(mdir / "method_diagnostics.json", "w") as f:
        json.dump(diag, f, indent=2, default=str)
    print(f"  -> {method_name}: bal_acc={diag['mean_balanced_accuracy']} | "
          f"collapse_rate={diag['collapse_rate']} | pos_class_pred_rate={diag['positive_class_pred_rate']}")

with open(ARTIFACT_DIR / "method_summary.json", "w") as f:
    json.dump(METHOD_SUMMARY, f, indent=2, default=str)


[2026-06-13 09:49:02] ======================================================================
[2026-06-13 09:49:02] METHOD: logreg_bandpower (sklearn_features)
[2026-06-13 09:49:02] ======================================================================
[2026-06-13 09:49:02]   subject 1: bal_acc=0.700±0.100
[2026-06-13 09:49:02]   subject 2: bal_acc=0.375±0.079
[2026-06-13 09:49:02]   subject 3: bal_acc=0.400±0.050
[2026-06-13 09:49:02]   subject 4: bal_acc=0.500±0.079
[2026-06-13 09:49:02]   subject 5: bal_acc=0.375±0.158
[2026-06-13 09:49:02]   subject 6: bal_acc=0.500±0.177
[2026-06-13 09:49:02]   subject 7: bal_acc=0.750±0.079
[2026-06-13 09:49:02]   subject 8: bal_acc=0.800±0.150
[2026-06-13 09:49:02]   subject 9: bal_acc=0.500±0.137
[2026-06-13 09:49:02]   subject 10: bal_acc=0.550±0.127
[2026-06-13 09:49:02]   subject 11: bal_acc=0.475±0.146
[2026-06-13 09:49:02]   subject 12: bal_acc=0.425±0.203
[2026-06-13 09:49:02]   subject 13: bal_acc=0.725±0.122
[2026-06-13 09:49:02]   subje

<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarni

[2026-06-13 09:50:07]   subject 32: FAILED (SubjectArrayDataset.__init__() missing 1 required positional argument: 'subject_id')
[2026-06-13 09:50:07]   subject 33: FAILED (SubjectArrayDataset.__init__() missing 1 required positional argument: 'subject_id')
[2026-06-13 09:50:07]   subject 34: FAILED (SubjectArrayDataset.__init__() missing 1 required positional argument: 'subject_id')
[2026-06-13 09:50:07]   subject 35: FAILED (SubjectArrayDataset.__init__() missing 1 required positional argument: 'subject_id')
[2026-06-13 09:50:07]   subject 36: FAILED (SubjectArrayDataset.__init__() missing 1 required positional argument: 'subject_id')
[2026-06-13 09:50:07]   subject 37: FAILED (SubjectArrayDataset.__init__() missing 1 required positional argument: 'subject_id')
[2026-06-13 09:50:07]   subject 38: FAILED (SubjectArrayDataset.__init__() missing 1 required positional argument: 'subject_id')
[2026-06-13 09:50:07]   subject 39: FAILED (SubjectArrayDataset.__init__() missing 1 required pos

<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarning: NOTE: EEGNetv4() is a deprecated class. `EEGNetv4` was renamed to `EEGNet` in v1.12; this alias will be removed in v1.14..
<decorator-gen-527>:4: FutureWarni

[2026-06-13 09:50:07]   subject 1: bal_acc=0.325±0.100
[2026-06-13 09:50:08]   subject 2: bal_acc=0.425±0.100
[2026-06-13 09:50:08]   subject 3: bal_acc=0.575±0.170
[2026-06-13 09:50:08]   subject 4: bal_acc=0.475±0.146
[2026-06-13 09:50:08]   subject 5: bal_acc=0.600±0.094
[2026-06-13 09:50:08]   subject 6: bal_acc=0.425±0.100
[2026-06-13 09:50:09]   subject 7: bal_acc=0.800±0.127
[2026-06-13 09:50:09]   subject 8: bal_acc=0.700±0.127
[2026-06-13 09:50:09]   subject 9: bal_acc=0.425±0.100
[2026-06-13 09:50:09]   subject 10: bal_acc=0.275±0.184
[2026-06-13 09:50:09]   subject 11: bal_acc=0.400±0.094
[2026-06-13 09:50:10]   subject 12: bal_acc=0.400±0.166
[2026-06-13 09:50:10]   subject 13: bal_acc=0.675±0.127
[2026-06-13 09:50:10]   subject 14: bal_acc=0.500±0.112
[2026-06-13 09:50:10]   subject 15: bal_acc=0.525±0.094
[2026-06-13 09:50:10]   subject 16: bal_acc=0.475±0.094
[2026-06-13 09:50:11]   subject 17: bal_acc=0.325±0.127
[2026-06-13 09:50:11]   subject 18: bal_acc=0.675±0.170
[

# 8. Cross-method comparison (Q3, Q5, Q7) + optional S-JEPA head-to-head

Tabulates the diagnostics per method. If `CONFIG["sjepa_reference_run_dir"]` points to a
prior S-JEPA run, its `global_metrics.json` is loaded as an extra row so you can see directly
whether the simple baselines match or beat the S-JEPA downstream head.

In [19]:
rows = []
for m, s in METHOD_SUMMARY.items():
    d = s["diagnostics"]; g = s["global_metrics"]
    rows.append({"method": m,
                 "mean_balanced_accuracy": d["mean_balanced_accuracy"],
                 "std_balanced_accuracy": d["std_balanced_accuracy"],
                 "mean_accuracy": g.get("mean_accuracy"),
                 "collapse_rate (Q3)": d["collapse_rate"],
                 "pos_class_pred_rate (Q7)": d["positive_class_pred_rate"],
                 "subject_balacc_std": d["subject_balacc_std"],
                 "subject_balacc_min": d["subject_balacc_min"],
                 "mean_confidence (Q8)": d["mean_confidence"],
                 "frac_conf>0.9 (Q8)": d["frac_confidence_over_0p9"]})

sjepa_dir = CONFIG.get("sjepa_reference_run_dir")
if sjepa_dir and Path(sjepa_dir, "global_metrics.json").exists():
    gm = json.load(open(Path(sjepa_dir, "global_metrics.json")))
    rows.append({"method": "sjepa_reference (loaded)",
                 "mean_balanced_accuracy": gm.get("mean_balanced_accuracy"),
                 "std_balanced_accuracy": gm.get("std_balanced_accuracy"),
                 "mean_accuracy": gm.get("mean_accuracy")})
    print(f"Loaded S-JEPA reference metrics from {sjepa_dir}")
else:
    print("No S-JEPA reference run loaded (set CONFIG['sjepa_reference_run_dir'] to compare).")

comparison_df = pd.DataFrame(rows).sort_values("mean_balanced_accuracy", ascending=False, na_position="last")
try:
    from IPython.display import display
    display(comparison_df)
except Exception:
    print(comparison_df.to_string(index=False))
comparison_df.to_csv(ARTIFACT_DIR / "method_comparison.csv", index=False)

if HAVE_MPL and len(comparison_df):
    plot_df = comparison_df.dropna(subset=["mean_balanced_accuracy"])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.barh(plot_df["method"], plot_df["mean_balanced_accuracy"],
            xerr=plot_df["std_balanced_accuracy"].fillna(0), capsize=4)
    ax.axvline(0.5, ls="--", c="grey", lw=1, label="chance")
    ax.set_xlabel("mean balanced accuracy"); ax.set_xlim(0, 1); ax.legend()
    ax.set_title("Baseline comparison (within-subject CV)")
    fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "method_comparison.png", dpi=160); plt.close(fig)
    print(f"Saved: {ARTIFACT_DIR / 'method_comparison.png'}")


[2026-06-13 09:50:18] No S-JEPA reference run loaded (set CONFIG['sjepa_reference_run_dir'] to compare).


,method,mean_balanced_accuracy,std_balanced_accuracy,mean_accuracy,collapse_rate (Q3),pos_class_pred_rate (Q7),subject_balacc_std,subject_balacc_min,mean_confidence (Q8),frac_conf>0.9 (Q8)
1,csp_lda,0.5500,0.212899,0.5500,0.026087,0.458696,0.136732,0.300,0.762486,0.307609
0,logreg_bandpower,0.5415,0.195487,0.5415,0.016000,0.505500,0.132534,0.325,0.727305,0.166000
2,fbcsp_lda,0.5325,0.175731,0.5325,0.008000,0.509500,0.080816,0.400,0.984659,0.952000
3,riemann_mdm,0.5185,0.202720,0.5185,0.040000,0.523500,0.147038,0.250,0.878892,0.652000


[2026-06-13 09:50:18] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-supervised-classifier-diagnostics/20260613_0948_d81691de/method_comparison.png


# 9. Per-subject difficulty and per-fold failures (Q4, Q9)

A subject × method matrix of mean balanced accuracy reveals subjects that are hard for *every*
baseline ("consistently impossible") versus method-specific failures.

In [20]:
subj_method = {}
for m, fold_results in ALL_RESULTS.items():
    by_sub = {}
    for r in fold_results:
        by_sub.setdefault(r["subject_id"], []).append(r["balanced_accuracy"])
    subj_method[m] = {sid: float(np.mean(v)) for sid, v in by_sub.items()}

heat_df = pd.DataFrame(subj_method).sort_index()
if not heat_df.empty:
    heat_df["mean_across_methods"] = heat_df.mean(axis=1)
    heat_df = heat_df.sort_values("mean_across_methods")
    try:
        from IPython.display import display
        display(heat_df.round(3))
    except Exception:
        print(heat_df.round(3).to_string())
    heat_df.to_csv(ARTIFACT_DIR / "subject_by_method_balacc.csv")

    hard = heat_df.index[heat_df["mean_across_methods"] <= 0.55].tolist()
    print(f"\nSubjects at/below 0.55 mean balanced accuracy across all methods "
          f"(candidate 'consistently hard'): {hard}")

    if HAVE_MPL:
        mat = heat_df.drop(columns=["mean_across_methods"])
        fig, ax = plt.subplots(figsize=(1.6 * mat.shape[1] + 2, 0.35 * mat.shape[0] + 2))
        im = ax.imshow(mat.values, aspect="auto", vmin=0.3, vmax=0.9, cmap="viridis")
        ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=30, ha="right")
        ax.set_yticks(range(mat.shape[0])); ax.set_yticklabels(mat.index)
        ax.set_title("Mean balanced accuracy: subject x method")
        fig.colorbar(im, ax=ax, label="balanced accuracy")
        fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "subject_by_method_heatmap.png", dpi=160); plt.close(fig)
        print(f"Saved: {ARTIFACT_DIR / 'subject_by_method_heatmap.png'}")


,logreg_bandpower,csp_lda,fbcsp_lda,riemann_mdm,mean_across_methods
35,0.525,0.300,0.575,0.250,0.412
19,0.350,NaN,0.550,0.400,0.433
31,0.375,0.375,0.600,0.400,0.438
11,0.475,NaN,0.475,0.400,0.450
12,0.425,NaN,0.525,0.400,0.450
6,0.500,NaN,0.425,0.425,0.450
21,0.450,NaN,0.625,0.275,0.450
46,0.350,NaN,0.600,0.400,0.450
2,0.375,0.475,0.550,0.425,0.456
10,0.550,0.450,0.550,0.275,0.456



[2026-06-13 09:50:18] Subjects at/below 0.55 mean balanced accuracy across all methods (candidate 'consistently hard'): ['35', '19', '31', '11', '12', '6', '21', '46', '2', '10', '33', '48', '17', '49', '4', '5', '27', '42', '15', '36', '9', '1', '44', '23', '25', '41', '3', '32', '24', '47', '26', '16', '43', '29', '14', '34', '39']
[2026-06-13 09:50:18] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-supervised-classifier-diagnostics/20260613_0948_d81691de/subject_by_method_heatmap.png


# 10. Window-start sensitivity (Q6)

Re-preprocesses each subject at several `mi_window_start_s` values (everything else held
fixed) and re-evaluates the cheap linear baseline, to show whether decoding depends strongly
on *when* the analysis window starts. Re-preprocessing is explicit (no silent change) and the
original CONFIG is restored afterward.

In [21]:
ws_cfg = CONFIG.get("window_sensitivity", {})
WINDOW_SENSITIVITY_DF = None
if not ws_cfg.get("enabled", False):
    print("Window-start sensitivity disabled in CONFIG.")
elif not HAVE_MNE:
    print("MNE unavailable -> cannot re-preprocess for the window sweep.")
else:
    method_name = ws_cfg.get("method", "logreg_bandpower")
    spec = CONFIG["methods"][method_name]
    grid = ws_cfg.get("mi_window_start_grid_s", [1.0, 1.5, 2.0])
    sids = sorted(SUBJECT_ARRAYS.keys(), key=_sort_subject_key)
    if ws_cfg.get("max_subjects"):
        sids = sids[:int(ws_cfg["max_subjects"])]

    # cache raw arrays once
    raw_cache = {}
    for item in subjects_df.to_dict("records"):
        sid = str(int(item["subject_id"]))
        if sid in sids:
            Xr, yr, _, _ = load_subject_mat(Path(item["path"]))
            raw_cache[sid] = (Xr, yr)

    saved_start = CONFIG["mi_window_start_s"]
    rows = []
    try:
        for start_s in grid:
            stop_sample = int(round(start_s * EFFECTIVE_SFREQ)) + int(WINDOW_SAMPLES)
            resampled_len = int(round(LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL * EFFECTIVE_SFREQ / LIU_SOURCE_SFREQ))
            if stop_sample > resampled_len:
                print(f"  start={start_s}s skipped (window would exceed trial: {stop_sample}>{resampled_len})")
                continue
            CONFIG["mi_window_start_s"] = float(start_s)
            for sid in sids:
                Xr, yr = raw_cache[sid]
                Xw, yy, _, _, _ = preprocess_subject_configurable(Xr, yr, int(sid))
                res = evaluate_method_on_subject(method_name, spec, sid, Xw.astype(np.float64), yy.astype(int))
                bals = [r["balanced_accuracy"] for r in res]
                rows.append({"mi_window_start_s": float(start_s), "subject_id": sid,
                             "mean_balanced_accuracy": float(np.mean(bals))})
    finally:
        CONFIG["mi_window_start_s"] = saved_start  # restore (no silent change)

    WINDOW_SENSITIVITY_DF = pd.DataFrame(rows)
    if not WINDOW_SENSITIVITY_DF.empty:
        pivot = WINDOW_SENSITIVITY_DF.pivot(index="subject_id", columns="mi_window_start_s",
                                            values="mean_balanced_accuracy")
        print(f"Window-start sensitivity ({method_name}, balanced accuracy):")
        print(pivot.round(3).to_string())
        WINDOW_SENSITIVITY_DF.to_csv(ARTIFACT_DIR / "window_start_sensitivity.csv", index=False)
        if HAVE_MPL:
            fig, ax = plt.subplots(figsize=(7, 4))
            for sid in pivot.index:
                ax.plot(pivot.columns, pivot.loc[sid], color="grey", alpha=0.4, lw=1)
            ax.plot(pivot.columns, pivot.mean(axis=0), "o-", color="C0", lw=2, label="mean across subjects")
            ax.axhline(0.5, ls="--", c="red", lw=1, label="chance")
            ax.set_xlabel("MI window start (s)"); ax.set_ylabel("mean balanced accuracy")
            ax.set_ylim(0.3, 1.0); ax.legend(); ax.set_title(f"Window-start sensitivity ({method_name})")
            fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "window_start_sensitivity.png", dpi=160); plt.close(fig)
            print(f"Saved: {ARTIFACT_DIR / 'window_start_sensitivity.png'}")


[2026-06-13 09:51:22] Window-start sensitivity (logreg_bandpower, balanced accuracy):
[2026-06-13 09:51:22] mi_window_start_s    0.5    1.0    1.5    2.0    2.5    3.0
subject_id                                                 
1                  0.425  0.625  0.700  0.700  0.575  0.625
10                 0.450  0.500  0.550  0.425  0.525  0.450
11                 0.400  0.475  0.475  0.600  0.650  0.725
12                 0.350  0.450  0.425  0.400  0.500  0.500
13                 0.700  0.725  0.725  0.725  0.725  0.700
14                 0.750  0.625  0.650  0.475  0.525  0.525
15                 0.300  0.375  0.400  0.700  0.550  0.550
16                 0.600  0.600  0.550  0.425  0.525  0.475
17                 0.350  0.400  0.550  0.525  0.375  0.325
18                 0.650  0.650  0.650  0.700  0.625  0.550
19                 0.450  0.425  0.350  0.375  0.475  0.425
2                  0.525  0.400  0.375  0.450  0.500  0.525
20                 0.625  0.600  0.675  0.675  0.650

# 11. Probability calibration / saturation (Q8)

Aggregates predicted-class confidence across all folds of each probabilistic method into a
reliability curve (binned confidence vs empirical accuracy). A curve hugging the diagonal is
well-calibrated; mass piled near confidence 1.0 with the curve below it indicates
over-confident/saturated probabilities.

In [22]:
def calibration_for_method(method_name, fold_results, n_bins=10):
    """Per-sample reliability: bin by predicted confidence, compare to empirical accuracy.
    Test-label order is recovered by re-deriving the (deterministic) split for each fold."""
    edges = np.linspace(0, 1, n_bins + 1)
    bin_conf_sum = np.zeros(n_bins); bin_acc_sum = np.zeros(n_bins); bin_n = np.zeros(n_bins)
    for r in fold_results:
        pd_ = r.get("probability_diagnostics", {})
        if not pd_.get("available") or "_probs" not in pd_:
            continue
        probs = np.asarray(pd_["_probs"], dtype=float)
        sid, fid = r["subject_id"], r["fold_id"]
        X, y = SUBJECT_ARRAYS[sid]
        split = next(s for s in make_evaluation_splits(y, TARGET_N_CLASSES) if int(s["split_id"]) == int(fid))
        y_test = y[split["idx_test"]]
        if len(y_test) != len(probs):
            continue
        y_pred = probs.argmax(axis=1); conf = probs.max(axis=1)
        correct = (y_pred == y_test).astype(float)
        idx = np.clip(np.digitize(conf, edges) - 1, 0, n_bins - 1)
        for b in range(n_bins):
            m = idx == b
            if m.any():
                bin_conf_sum[b] += conf[m].sum(); bin_acc_sum[b] += correct[m].sum(); bin_n[b] += m.sum()
    valid = bin_n > 0
    return (bin_conf_sum[valid] / bin_n[valid], bin_acc_sum[valid] / bin_n[valid], bin_n[valid])

calib = {}
for m, fr in ALL_RESULTS.items():
    try:
        conf_x, acc_y, n = calibration_for_method(m, fr)
        if len(conf_x):
            calib[m] = (conf_x, acc_y, n)
    except Exception as exc:
        print(f"calibration for {m} failed: {exc}")

if calib and HAVE_MPL:
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    ax.plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
    for m, (cx, ay, n) in calib.items():
        ax.plot(cx, ay, "o-", label=m, alpha=0.8)
    ax.set_xlabel("mean predicted confidence"); ax.set_ylabel("empirical accuracy")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.set_title("Reliability curves")
    fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "calibration_reliability.png", dpi=160); plt.close(fig)
    print(f"Saved: {ARTIFACT_DIR / 'calibration_reliability.png'}")
else:
    print("No calibration curves (need probabilistic methods + matplotlib).")


[2026-06-13 09:51:22] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-supervised-classifier-diagnostics/20260613_0948_d81691de/calibration_reliability.png


## 12. Save run metadata

In [23]:
run_metadata = {
    "run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR),
    "experiment_name": CONFIG.get("experiment_name"), "config_note": CONFIG.get("config_note"),
    "purpose": "supervised baseline diagnostics (no S-JEPA)",
    "methods_run": list(METHOD_SUMMARY.keys()),
    "conservative_overrides": {"prediction_balance_loss_weight": CONFIG["prediction_balance_loss_weight"],
                                "augmentation_enabled": CONFIG["augmentation"]["enabled"]},
    "preprocessing_config": PREPROCESSING_CONFIG, "evaluation_config": EVALUATION_CONFIG,
    "effective_sfreq": EFFECTIVE_SFREQ, "window_samples": WINDOW_SAMPLES,
    "mi_window_start_s": CONFIG["mi_window_start_s"], "channel_names": list(CH_NAMES),
    "subjects": [str(s) for s in SUBJECTS] if "SUBJECTS" in globals() else None,
    "sjepa_reference_run_dir": CONFIG.get("sjepa_reference_run_dir"),
    "method_summary": METHOD_SUMMARY, "seed": int(BASE_SEED),
}
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)
print(f"Saved run metadata to: {ARTIFACT_DIR / 'run_metadata.json'}")
print("Artifacts:")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  - {p.name}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass


[2026-06-13 09:51:22] Saved run metadata to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-supervised-classifier-diagnostics/20260613_0948_d81691de/run_metadata.json
[2026-06-13 09:51:22] Artifacts:
[2026-06-13 09:51:22]   - calibration_reliability.png
[2026-06-13 09:51:22]   - config.json
[2026-06-13 09:51:22]   - label_split_sanity.csv
[2026-06-13 09:51:22]   - mat_structure_preview_first_subject.csv
[2026-06-13 09:51:22]   - method_comparison.csv
[2026-06-13 09:51:22]   - method_comparison.png
[2026-06-13 09:51:22]   - method_csp_lda
[2026-06-13 09:51:22]   - method_fbcsp_lda
[2026-06-13 09:51:22]   - method_logreg_bandpower
[2026-06-13 09:51:22]   - method_riemann_mdm
[2026-06-13 09:51:22]   - method_summary.json
[2026-06-13 09:51:22]   - run.log
[2026-06-13 09:51:22]   - run_metadata.json
[2026-06-13 09:51:22]   - subject_by_method_balacc.csv
[2026-06-13 09:51:22]   - subject_by_method_heatmap.png
[2026-06-13 09:51:22]   - subje

# 13. How to read these diagnostics

- **Q1 split balance / Q2 labels** — Section 5 should show two classes per subject with the
  {1,2}→{0,1} mapping and `All folds ... per class: True`. If not, stop here: a modeling
  result on top of a broken split is meaningless.
- **Q3 collapse / Q7 one-class** — In Sections 7-8, `collapse_rate` near 0 and
  `pos_class_pred_rate` near 0.5 are healthy. A simple, well-behaved baseline that does *not*
  collapse while the S-JEPA head does points at the model/optimization, not the data.
- **Q5 simple vs S-JEPA** — Set `CONFIG["sjepa_reference_run_dir"]` to your S-JEPA run folder;
  Section 8 then lists it alongside the baselines. If `csp_lda` / `logreg_bandpower` match or
  beat it, the deep model is not yet earning its complexity on this dataset.
- **Q4 hard subjects / Q9 fold failures** — Section 9's subject×method heatmap separates
  "hard for everyone" (a data/subject property) from "hard for one method" (a model property).
- **Q6 window dependence** — Section 10 shows balanced accuracy vs window start. A strong peak
  at a particular start is exactly the motor-imagery time-localization that time-window search
  (TWFB) exploits, and it tells you where to center S-JEPA windows.
- **Q8 calibration** — Section 11's reliability curves: points below the diagonal at high
  confidence mean over-confident/saturated probabilities (common when a model collapses).

**Conservative-by-design reminders.** Augmentation is off (toggle in `CONFIG["augmentation"]`);
the prediction-balance loss penalty is off so collapse is visible; preprocessing is the
reference pipeline unchanged; CSP, covariance, and feature scalers are fit on the training
fold only. The EEGNet settings are modest defaults for a sanity check, not a tuned model — do
not read its absolute number as the best a CNN can do here.